# Feature Engineering v3: Aggregate First, Join Contact Last (Dependency-Safe)

This notebook rewrites the original feature engineering pipeline to fix issues found during EDA:

1. Avoid early data loss by aggregating activity first and joining contact metadata only after developer-level features are built.
2. Handle zero-inflated behavior with binary flags plus `LOG1P` features.
3. Replace nested cumulative windows with non-overlapping recency windows: `0_30d`, `30_90d`, and `90_180d`.
4. Add recency, velocity, rate, and interaction features.
5. Normalize persona lane scores and add entropy so mixed personas are represented more honestly.
6. Add feature validation checks after major table creation.
7. Incorporate the AI-scraped effort mapping from `Activity_Score_Mapping_filled.xlsx` as a separate row-level effort signal.
8. Prevent event double counting by collapsing event type, attendance, and role into one composite row-level effort estimate instead of summing separate event components.
9. Add effort-vs-score gap features so activity score and user effort are no longer treated as one-to-one.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH = "developer_project.duckdb"
con = duckdb.connect(DB_PATH)

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 100)
 
ACTIVITY_TABLE = "activity_final"
CONTACT_TABLE = "contact_final"

# Robust helper: use natural log of 1+x in DuckDB.
LOG1P = "LN(1 + {x})"
EFFORT_MAPPING_PATH = Path("Data/Activity_Score_Mapping_filled.xlsx")
EFFORT_MAPPING_SHEET = "Activity_Score_Mapping"


## 1. Confirm source tables and columns

In [3]:
tables = con.execute("SHOW TABLES").fetchdf()
display(tables)

available = set(tables.iloc[:, 0].astype(str))
missing = {ACTIVITY_TABLE, CONTACT_TABLE} - available
if missing:
    raise ValueError(f"Missing required table(s): {missing}")

def get_columns(table_name: str) -> set:
    return set(con.execute(f"DESCRIBE {table_name}").fetchdf()["column_name"].astype(str))

activity_cols = get_columns(ACTIVITY_TABLE)
contact_cols = get_columns(CONTACT_TABLE)

required_activity_cols = {"dev_contact", "activity_date", "activity"}
required_contact_cols = {"developer_id"}

if required_activity_cols - activity_cols:
    raise ValueError(f"Missing activity columns: {required_activity_cols - activity_cols}")
if required_contact_cols - contact_cols:
    raise ValueError(f"Missing contact columns: {required_contact_cols - contact_cols}")

for table in [ACTIVITY_TABLE, CONTACT_TABLE]:
    print(f"\n{table}")
    display(con.execute(f"SELECT COUNT(*) AS rows FROM {table}").fetchdf())
    display(con.execute(f"DESCRIBE {table}").fetchdf())

,name
0,activity_base
1,activity_base_v2
2,activity_clean
3,activity_dictionary_v2
4,activity_enriched_v1
5,activity_final
6,activity_labeled_v2
7,activity_ontology_v1
8,activity_raw
9,activity_score_mapping_clean



activity_final


,rows
0,69347501


,column_name,column_type,null,key,default,extra
0,dev_contact,VARCHAR,YES,None,None,None
1,activity,VARCHAR,YES,None,None,None
2,activity_name,VARCHAR,YES,None,None,None
3,activity_type,VARCHAR,YES,None,None,None
4,activity_role,VARCHAR,YES,None,None,None
5,activity_attendance,VARCHAR,YES,None,None,None
6,activity_score,DOUBLE,YES,None,None,None
7,activity_date,DATE,YES,None,None,None
8,activity_id,VARCHAR,YES,None,None,None
9,filepath,VARCHAR,YES,None,None,None



contact_final


,rows
0,17806394


,column_name,column_type,null,key,default,extra
0,developer_id,VARCHAR,YES,None,None,None
1,program_application_source,VARCHAR,YES,None,None,None
2,country,VARCHAR,YES,None,None,None
3,region,VARCHAR,YES,None,None,None
4,sub_region,VARCHAR,YES,None,None,None
5,zone,VARCHAR,YES,None,None,None
6,territory,VARCHAR,YES,None,None,None
7,organization_english_name,VARCHAR,YES,None,None,None
8,development_areas,VARCHAR,YES,None,None,None
9,other_development_areas,VARCHAR,YES,None,None,None


## 2. Source sanity checks

In [4]:
dup_name_col = "activity_name" if "activity_name" in activity_cols else "activity"
source_validation = con.execute(f"""
WITH activity_checks AS (
    SELECT
        COUNT(*) AS activity_rows,
        COUNT(DISTINCT CAST(dev_contact AS VARCHAR)) AS activity_developers,
        SUM(CASE WHEN dev_contact IS NULL OR TRIM(CAST(dev_contact AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_dev_contact,
        SUM(CASE WHEN activity_date IS NULL THEN 1 ELSE 0 END) AS missing_activity_date,
        SUM(CASE WHEN activity_date > CURRENT_DATE THEN 1 ELSE 0 END) AS future_activity_dates,
        COUNT(*) - COUNT(DISTINCT (
            COALESCE(CAST(dev_contact AS VARCHAR), '') || '|' ||
            COALESCE(CAST(activity_date AS VARCHAR), '') || '|' ||
            COALESCE(CAST(activity AS VARCHAR), '') || '|' ||
            COALESCE(CAST({dup_name_col} AS VARCHAR), '')
        )) AS possible_duplicate_activity_rows
    FROM {ACTIVITY_TABLE}
),
contact_checks AS (
    SELECT
        COUNT(*) AS contact_rows,
        COUNT(DISTINCT CAST(developer_id AS VARCHAR)) AS contact_developers,
        SUM(CASE WHEN developer_id IS NULL OR TRIM(CAST(developer_id AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_developer_id,
        COUNT(*) - COUNT(DISTINCT CAST(developer_id AS VARCHAR)) AS duplicate_contact_rows
    FROM {CONTACT_TABLE}
)
SELECT * FROM activity_checks CROSS JOIN contact_checks
""").fetchdf()

display(source_validation)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,activity_rows,activity_developers,missing_dev_contact,missing_activity_date,future_activity_dates,possible_duplicate_activity_rows,contact_rows,contact_developers,missing_developer_id,duplicate_contact_rows
0,69347501,7660278,0.0,0.0,0.0,33695697,17806394,8903197,0.0,8903197


## 3. Load AI-generated effort mapping

The mapping file is treated as enrichment, not as a replacement for NVIDIA's original `activity_score`. It creates a separate effort axis. For event rows, multiple possible matches such as event type, attendance status, and role are collapsed later into one row-level effort value so registrations, attendance, and event type are not double counted.


In [5]:
if not EFFORT_MAPPING_PATH.exists():
    raise FileNotFoundError(
        f"Missing {EFFORT_MAPPING_PATH}. Place Activity_Score_Mapping_filled.xlsx in the notebook working directory."
    )

raw_effort_map = pd.read_excel(EFFORT_MAPPING_PATH, sheet_name=EFFORT_MAPPING_SHEET)

required_effort_cols = {"object", "field", "value", "score", "Effort Level", "Confidence"}
missing_effort_cols = required_effort_cols - set(raw_effort_map.columns)
if missing_effort_cols:
    raise ValueError(f"Missing required effort mapping columns: {missing_effort_cols}")

def clean_text(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .fillna("")
         .str.strip()
         .str.lower()
         .str.replace(r"\s+", " ", regex=True)
    )

effort_level_rank = {
    "passive": 0,
    "low": 1,
    "moderate": 2,
    "high": 3,
    "very high": 4,
}
confidence_weight = {
    "high": 1.00,
    "medium": 0.75,
    "low": 0.50,
    "": 0.50,
}

effort_map = raw_effort_map.copy()
effort_map["mapping_object_norm"] = clean_text(effort_map["object"])
effort_map["mapping_field_norm"] = clean_text(effort_map["field"])
effort_map["mapping_value_norm"] = clean_text(effort_map["value"])
effort_map["ai_effort_level"] = clean_text(effort_map["Effort Level"])
effort_map["ai_confidence"] = clean_text(effort_map["Confidence"])
effort_map["ai_activity_score_guideline"] = pd.to_numeric(effort_map["score"], errors="coerce")
effort_map["ai_effort_rank"] = effort_map["ai_effort_level"].map(effort_level_rank).fillna(np.nan)
effort_map["ai_confidence_weight"] = effort_map["ai_confidence"].map(confidence_weight).fillna(0.50)
effort_map["ai_weighted_effort_rank"] = effort_map["ai_effort_rank"] * effort_map["ai_confidence_weight"]

effort_map_for_duckdb = effort_map[[
    "mapping_object_norm", "mapping_field_norm", "mapping_value_norm",
    "ai_effort_level", "ai_effort_rank", "ai_weighted_effort_rank",
    "ai_confidence", "ai_confidence_weight", "ai_activity_score_guideline",
    "Notes", "Other Notes"
]].rename(columns={"Notes": "ai_effort_notes", "Other Notes": "ai_effort_other_notes"})

con.register("effort_map_for_duckdb", effort_map_for_duckdb)
con.execute("CREATE OR REPLACE TABLE activity_effort_mapping_ai_v2 AS SELECT * FROM effort_map_for_duckdb")
con.unregister("effort_map_for_duckdb")

print("AI effort mapping loaded")
display(con.execute("""
SELECT
    COUNT(*) AS mapping_rows,
    COUNT(DISTINCT mapping_object_norm) AS mapping_objects,
    SUM(CASE WHEN ai_effort_rank IS NULL THEN 1 ELSE 0 END) AS missing_effort_rank_rows,
    ROUND(AVG(ai_confidence_weight), 3) AS avg_confidence_weight
FROM activity_effort_mapping_ai_v2
""").fetchdf())

display(con.execute("""
SELECT mapping_object_norm, mapping_field_norm, ai_effort_level, ai_confidence, COUNT(*) AS rows
FROM activity_effort_mapping_ai_v2
GROUP BY 1,2,3,4
ORDER BY mapping_object_norm, mapping_field_norm, rows DESC
""").fetchdf())


AI effort mapping loaded


,mapping_rows,mapping_objects,missing_effort_rank_rows,avg_confidence_weight
0,39,7,1.0,0.865


,mapping_object_norm,mapping_field_norm,ai_effort_level,ai_confidence,rows
0,dev dli attendance,course type,high,high,2
1,dev dli attendance,course type,moderate,medium,1
2,dev dli attendance,course type,moderate,high,1
3,dev event,event type,moderate,high,2
4,dev event,event type,low,low,1
5,dev event,event type,high,high,1
6,dev event attendance,role,very high,high,1
7,dev event attendance,status,low,high,2
8,dev event attendance,status,moderate,high,1
9,dev file,file type,high,high,3


## 3. Build an activity-only base table

Important design change: this table does **not** join contact metadata. Activity features are created from activity rows first so unmatched contacts do not cause early population loss and duplicate contact rows do not multiply activity rows.

In [6]:
def activity_col_expr(col: str, out: str = None, cast: str = "VARCHAR", default: str = "NULL") -> str:
    out = out or col
    if col in activity_cols:
        return f"CAST(a.{col} AS {cast}) AS {out}"
    return f"CAST({default} AS {cast}) AS {out}"

activity_score_expr = (
    "LEAST(GREATEST(COALESCE(TRY_CAST(a.activity_score AS DOUBLE), 0.0), 0.0), 100.0) AS activity_score"
    if "activity_score" in activity_cols else
    "CAST(0.0 AS DOUBLE) AS activity_score"
)

con.execute(f"""
CREATE OR REPLACE TABLE activity_base_v2 AS
SELECT
    CAST(a.dev_contact AS VARCHAR) AS developer_id,
    CAST(a.activity_date AS DATE) AS activity_date,
    LOWER(TRIM(CAST(a.activity AS VARCHAR))) AS activity,
    {activity_col_expr('activity_name')},
    {activity_col_expr('activity_type')},
    {activity_col_expr('activity_role')},
    {activity_col_expr('activity_attendance')},
    {activity_score_expr},
    {activity_col_expr('filepath')},
    {activity_col_expr('lead_source')},
    {activity_col_expr('lead_source_details')},
    {activity_col_expr('nvidia_campaign_id')}
FROM {ACTIVITY_TABLE} a
WHERE a.dev_contact IS NOT NULL
  AND TRIM(CAST(a.dev_contact AS VARCHAR)) <> ''
  AND a.activity_date IS NOT NULL
""")

display(con.execute("""
SELECT
    (SELECT COUNT(*) FROM activity_final WHERE dev_contact IS NOT NULL AND TRIM(CAST(dev_contact AS VARCHAR)) <> '' AND activity_date IS NOT NULL) AS valid_source_activity_rows,
    COUNT(*) AS activity_base_rows,
    COUNT(DISTINCT developer_id) AS activity_base_developers,
    MIN(activity_date) AS min_activity_date,
    MAX(activity_date) AS max_activity_date,
    MIN(activity_score) AS min_activity_score,
    MAX(activity_score) AS max_activity_score
FROM activity_base_v2
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,valid_source_activity_rows,activity_base_rows,activity_base_developers,min_activity_date,max_activity_date,min_activity_score,max_activity_score
0,69347501,69347501,7660278,2020-01-01,2026-03-12,0.0,100.0


## 4. Deduplicate contact metadata separately

This table is used only for final enrichment and profile-text persona hints. It is not joined into activity rows.

In [7]:
def contact_select_expr(col: str, out: str = None, cast: str = "VARCHAR", default: str = "NULL") -> str:
    out = out or col
    if col in contact_cols:
        return f"CAST({col} AS {cast}) AS {out}"
    return f"CAST({default} AS {cast}) AS {out}"

order_terms = []
if "last_modified_date" in contact_cols:
    order_terms.append("last_modified_date DESC NULLS LAST")
if "created_date" in contact_cols:
    order_terms.append("created_date DESC NULLS LAST")
order_by = ", ".join(order_terms) if order_terms else "developer_id"

contact_fields = [
    "developer_id", "created_date", "first_activity_date", "last_activity_date",
    "development_areas", "fields_of_interest", "account_id", "account_type",
    "country", "region", "industry_segment_vertical", "program_application_source",
    "organization_english_name", "normalized_account_name", "wwfo_category", "wwfo_target_list"
]

select_list = []
for col in contact_fields:
    if col == "developer_id":
        select_list.append("CAST(developer_id AS VARCHAR) AS developer_id")
    elif col in {"created_date", "first_activity_date", "last_activity_date"} and col in contact_cols:
        select_list.append(f"CAST({col} AS DATE) AS {col}")
    else:
        select_list.append(contact_select_expr(col))

con.execute(f"""
CREATE OR REPLACE TABLE contact_one_row_v2 AS
WITH ranked AS (
    SELECT
        {', '.join(select_list)},
        ROW_NUMBER() OVER (PARTITION BY CAST(developer_id AS VARCHAR) ORDER BY {order_by}) AS rn
    FROM {CONTACT_TABLE}
    WHERE developer_id IS NOT NULL
      AND TRIM(CAST(developer_id AS VARCHAR)) <> ''
)
SELECT * EXCLUDE (rn)
FROM ranked
WHERE rn = 1
""")

display(con.execute("""
SELECT
    COUNT(*) AS contact_one_row_rows,
    COUNT(DISTINCT developer_id) AS contact_one_row_developers
FROM contact_one_row_v2
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,contact_one_row_rows,contact_one_row_developers
0,8903197,8903197


## 6. Build deterministic activity dictionary, then enrich rows with AI effort mapping

The dictionary remains as a conservative fallback. The AI effort mapping is then matched to row-level fields such as `activity_type`, `activity_attendance`, and `activity_role`.

For event-like rows, event type, attendance status, and role are collapsed into one candidate row-level effort estimate. We take the strongest matched signal rather than adding all of them, which prevents double counting when the same event appears through registration, attendance, and event metadata.


In [8]:
# Deterministic activity dictionary: one row per activity.
# This is the conservative fallback when the AI effort mapping cannot match a row.
con.execute(r"""
CREATE OR REPLACE TABLE activity_dictionary_v2 AS
WITH activities AS (
    SELECT DISTINCT activity
    FROM activity_base_v2
    WHERE activity IS NOT NULL
)
SELECT
    activity,

    CASE
        -- Contribution / advocacy style signals.
        WHEN activity IN ('forum contributions') THEN 'Champion'
        WHEN activity IN ('user feedback', 'bugs filed') THEN 'Evaluate'

        -- Product / implementation behavior.
        WHEN activity IN ('hosted api', 'brev', 'ngc downloads', 'devzone downloads', 'sdk downloads') THEN 'Build'

        -- Formal learning.
        WHEN activity IN ('dli training') THEN 'Learn'
        WHEN activity IN ('webinars', 'on-demand views', 'conf. sessions live') THEN 'Learn'

        -- Events, membership, campaigns, and lightweight engagement.
        WHEN activity IN ('conference', 'other events', 'event registrations', 'program applications',
                          'dev program membership', 'product specific comms', 'contests') THEN 'Discover'

        -- Conservative fallback.
        ELSE 'Discover'
    END AS fallback_journey_signal,

    CASE
        WHEN activity IN ('forum contributions', 'hosted api', 'brev', 'ngc downloads', 'devzone downloads',
                          'sdk downloads', 'user feedback', 'bugs filed') THEN 'high'
        WHEN activity IN ('dli training', 'webinars', 'on-demand views', 'conf. sessions live',
                          'conference', 'other events', 'contests') THEN 'moderate'
        ELSE 'passive'
    END AS fallback_effort_level,

    CASE
        WHEN activity IN ('forum contributions', 'hosted api', 'brev', 'ngc downloads', 'devzone downloads',
                          'sdk downloads', 'user feedback', 'bugs filed') THEN 3.0
        WHEN activity IN ('dli training', 'webinars', 'on-demand views', 'conf. sessions live',
                          'conference', 'other events', 'contests') THEN 2.0
        ELSE 0.0
    END AS fallback_effort_rank,

    CASE
        WHEN activity IN ('ngc downloads', 'devzone downloads', 'sdk downloads') THEN 'Download'
        WHEN activity IN ('hosted api') THEN 'Hosted API'
        WHEN activity IN ('brev') THEN 'Cloud Workspace'
        WHEN activity IN ('forum contributions') THEN 'Community'
        WHEN activity IN ('dli training') THEN 'Training'
        WHEN activity IN ('webinars', 'conference', 'other events', 'event registrations', 'conf. sessions live') THEN 'Event'
        ELSE 'Content'
    END AS modality
FROM activities
""")

# Join the dictionary to activity facts and enrich with row-level AI effort mapping.
# Effort is now a separate axis from activity_score. The score-effort gap is retained as a model feature.
con.execute(r"""
CREATE OR REPLACE TABLE activity_labeled_v2 AS
WITH base AS (
    SELECT
        a.*,
        d.fallback_journey_signal,
        d.fallback_effort_level,
        d.fallback_effort_rank,
        d.modality,
        LOWER(TRIM(COALESCE(a.activity_type, ''))) AS activity_type_norm,
        LOWER(TRIM(COALESCE(a.activity_attendance, ''))) AS activity_attendance_norm,
        LOWER(TRIM(COALESCE(a.activity_role, ''))) AS activity_role_norm,
        LOWER(TRIM(COALESCE(a.activity_name, ''))) AS activity_name_norm,
        LOWER(TRIM(COALESCE(a.filepath, ''))) AS filepath_norm,
        LOWER(
            COALESCE(a.activity_name, '') || ' ' ||
            COALESCE(a.filepath, '') || ' ' ||
            COALESCE(a.lead_source_details, '') || ' ' ||
            COALESCE(a.activity_type, '') || ' ' ||
            COALESCE(a.activity_role, '') || ' ' ||
            COALESCE(a.lead_source, '')
        ) AS persona_activity_text
    FROM activity_base_v2 a
    LEFT JOIN activity_dictionary_v2 d USING (activity)
), candidates AS (
    -- Event type / session type candidate.
    SELECT
        b.developer_id, b.activity_date, b.activity, b.activity_name, b.activity_type, b.activity_role,
        b.activity_attendance, b.activity_score, b.filepath, b.lead_source, b.lead_source_details, b.nvidia_campaign_id,
        b.fallback_journey_signal, b.fallback_effort_level, b.fallback_effort_rank, b.modality, b.persona_activity_text,
        m.ai_effort_level, m.ai_effort_rank, m.ai_weighted_effort_rank, m.ai_confidence,
        m.ai_confidence_weight, m.ai_activity_score_guideline,
        'event_or_dli_or_file_type' AS ai_effort_match_source
    FROM base b
    JOIN activity_effort_mapping_ai_v2 m
      ON m.mapping_value_norm = b.activity_type_norm
     AND m.mapping_field_norm IN ('event type', 'course type', 'file type', 'form type', 'post type', 'status')
     AND (
            (b.activity IN ('conference', 'other events', 'conf. sessions live', 'webinars', 'on-demand views')
                AND m.mapping_object_norm IN ('dev event', 'dev event attendance'))
         OR (b.activity = 'dli training' AND m.mapping_object_norm = 'dev dli attendance')
         OR (b.activity IN ('devzone downloads', 'ngc downloads', 'sdk downloads') AND m.mapping_object_norm = 'dev file')
         OR (b.activity IN ('program applications', 'dev program membership') AND m.mapping_object_norm = 'dev program applications')
         OR (b.activity IN ('user feedback', 'event registrations') AND m.mapping_object_norm = 'dev form')
         OR (b.activity = 'forum contributions' AND m.mapping_object_norm = 'dev forum activities')
     )

    UNION ALL

    -- Attendance status candidate. This competes with event type; it is not added to it.
    SELECT
        b.developer_id, b.activity_date, b.activity, b.activity_name, b.activity_type, b.activity_role,
        b.activity_attendance, b.activity_score, b.filepath, b.lead_source, b.lead_source_details, b.nvidia_campaign_id,
        b.fallback_journey_signal, b.fallback_effort_level, b.fallback_effort_rank, b.modality, b.persona_activity_text,
        m.ai_effort_level, m.ai_effort_rank, m.ai_weighted_effort_rank, m.ai_confidence,
        m.ai_confidence_weight, m.ai_activity_score_guideline,
        'attendance_status' AS ai_effort_match_source
    FROM base b
    JOIN activity_effort_mapping_ai_v2 m
      ON m.mapping_value_norm = b.activity_attendance_norm
     AND m.mapping_object_norm IN ('dev event attendance', 'dev dli attendance')
     AND m.mapping_field_norm IN ('status', 'course type')

    UNION ALL

    -- Speaker/presenter or other role candidate. This can outrank regular event attendance.
    SELECT
        b.developer_id, b.activity_date, b.activity, b.activity_name, b.activity_type, b.activity_role,
        b.activity_attendance, b.activity_score, b.filepath, b.lead_source, b.lead_source_details, b.nvidia_campaign_id,
        b.fallback_journey_signal, b.fallback_effort_level, b.fallback_effort_rank, b.modality, b.persona_activity_text,
        m.ai_effort_level, m.ai_effort_rank, m.ai_weighted_effort_rank, m.ai_confidence,
        m.ai_confidence_weight, m.ai_activity_score_guideline,
        'role' AS ai_effort_match_source
    FROM base b
    JOIN activity_effort_mapping_ai_v2 m
      ON m.mapping_value_norm = b.activity_role_norm
     AND m.mapping_object_norm = 'dev event attendance'
     AND m.mapping_field_norm = 'role'
), best_candidate AS (
    SELECT * EXCLUDE (rn)
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY developer_id, activity_date, activity, activity_name, activity_type,
                             activity_role, activity_attendance, filepath, lead_source, lead_source_details,
                             nvidia_campaign_id, activity_score
                ORDER BY ai_effort_rank DESC NULLS LAST,
                         ai_confidence_weight DESC NULLS LAST,
                         ai_activity_score_guideline DESC NULLS LAST
            ) AS rn
        FROM candidates
    ) x
    WHERE rn = 1
), enriched AS (
    SELECT
        b.*,
        bc.ai_effort_level,
        bc.ai_effort_rank,
        bc.ai_weighted_effort_rank,
        bc.ai_confidence,
        bc.ai_confidence_weight,
        bc.ai_activity_score_guideline,
        bc.ai_effort_match_source
    FROM base b
    LEFT JOIN best_candidate bc
      ON COALESCE(b.developer_id, '') = COALESCE(bc.developer_id, '')
     AND COALESCE(CAST(b.activity_date AS VARCHAR), '') = COALESCE(CAST(bc.activity_date AS VARCHAR), '')
     AND COALESCE(b.activity, '') = COALESCE(bc.activity, '')
     AND COALESCE(b.activity_name, '') = COALESCE(bc.activity_name, '')
     AND COALESCE(b.activity_type, '') = COALESCE(bc.activity_type, '')
     AND COALESCE(b.activity_role, '') = COALESCE(bc.activity_role, '')
     AND COALESCE(b.activity_attendance, '') = COALESCE(bc.activity_attendance, '')
     AND COALESCE(b.filepath, '') = COALESCE(bc.filepath, '')
     AND COALESCE(b.lead_source, '') = COALESCE(bc.lead_source, '')
     AND COALESCE(b.lead_source_details, '') = COALESCE(bc.lead_source_details, '')
     AND COALESCE(b.nvidia_campaign_id, '') = COALESCE(bc.nvidia_campaign_id, '')
     AND COALESCE(b.activity_score, -999999) = COALESCE(bc.activity_score, -999999)
), finalized AS (
    SELECT
        * EXCLUDE (fallback_journey_signal, fallback_effort_level, fallback_effort_rank,
                   activity_type_norm, activity_attendance_norm, activity_role_norm, activity_name_norm, filepath_norm),
        CASE
            WHEN activity = 'devzone downloads' THEN
                CASE
                    WHEN filepath_norm LIKE '%.exe'
                      OR filepath_norm LIKE '%installer%'
                      OR filepath_norm LIKE '%toolkit%'
                      OR filepath_norm LIKE '%.deb'
                      OR filepath_norm LIKE '%.rpm'
                    THEN 'Build'
                    WHEN filepath_norm LIKE '%.pdf'
                      OR filepath_norm LIKE '%docs/%'
                      OR filepath_norm LIKE '%documentation%'
                    THEN 'Discover'
                    ELSE 'Evaluate'
                END
            WHEN LOWER(COALESCE(ai_effort_level, '')) = 'very high' THEN 'Champion'
            ELSE fallback_journey_signal
        END AS journey_signal,
        COALESCE(ai_effort_level, fallback_effort_level) AS effort_level,
        COALESCE(ai_effort_rank, fallback_effort_rank, 0.0) AS effort_rank,
        COALESCE(ai_weighted_effort_rank, fallback_effort_rank, 0.0) AS confidence_weighted_effort_rank,
        CASE WHEN COALESCE(ai_confidence, '') IN ('low', 'medium') THEN 1 ELSE 0 END AS low_medium_confidence_effort_flag,
        CASE WHEN ABS(COALESCE(activity_score, 0) - (COALESCE(ai_effort_rank, fallback_effort_rank, 0.0) * 25.0)) >= 50 THEN 1 ELSE 0 END AS score_effort_misalignment_flag,
        COALESCE(activity_score, 0) - (COALESCE(ai_effort_rank, fallback_effort_rank, 0.0) * 25.0) AS score_effort_gap,
        COALESCE(activity_score, 0) * COALESCE(ai_weighted_effort_rank, fallback_effort_rank, 0.0) AS effort_x_activity_score
    FROM enriched
), scored AS (
    SELECT
        *,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'cuda|cudnn|rapids|nccl|cutlass|\bdali\b|gpu|accelerated|hpc') THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END AS cuda_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'triton|tensorrt|nemo|\bnim\b|llm|large language|genai|generative|inference|model') THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END AS genai_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'isaac|robot|ros|autonomous machine|jetson|edge ai|autonomous') THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END AS robotics_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'omniverse|simulation|digital twin|modulus|render|rtx|graphics') THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END AS simulation_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'dli|training|course|workshop|webinar|certification|learn|community') THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END AS learning_community_activity_score
    FROM finalized
)
SELECT
    *,
    CASE
        WHEN GREATEST(cuda_activity_score, genai_activity_score, robotics_activity_score, simulation_activity_score, learning_community_activity_score) = 0 THEN 'Other'
        WHEN cuda_activity_score >= GREATEST(genai_activity_score, robotics_activity_score, simulation_activity_score, learning_community_activity_score) THEN 'CUDA'
        WHEN genai_activity_score >= GREATEST(cuda_activity_score, robotics_activity_score, simulation_activity_score, learning_community_activity_score) THEN 'GenAI'
        WHEN robotics_activity_score >= GREATEST(cuda_activity_score, genai_activity_score, simulation_activity_score, learning_community_activity_score) THEN 'Robotics'
        WHEN simulation_activity_score >= GREATEST(cuda_activity_score, genai_activity_score, robotics_activity_score, learning_community_activity_score) THEN 'Simulation'
        ELSE 'Learning_Community'
    END AS persona_hint,
    cuda_activity_score AS cuda_persona_score,
    genai_activity_score AS genai_persona_score,
    robotics_activity_score AS robotics_persona_score,
    simulation_activity_score AS simulation_persona_score,
    learning_community_activity_score AS learning_community_persona_score
FROM scored
""")

print('Dictionary size and coverage')
display(con.execute("""
SELECT
    COUNT(*) AS dictionary_rows,
    COUNT(DISTINCT activity) AS distinct_activities,
    SUM(CASE WHEN fallback_journey_signal IS NULL OR fallback_effort_level IS NULL OR modality IS NULL THEN 1 ELSE 0 END) AS missing_labels
FROM activity_dictionary_v2
""").fetchdf())

print('AI effort mapping coverage')
display(con.execute("""
SELECT
    COUNT(*) AS activity_rows,
    SUM(CASE WHEN ai_effort_level IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_ai_effort,
    ROUND(100.0 * SUM(CASE WHEN ai_effort_level IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_rows_with_ai_effort,
    SUM(low_medium_confidence_effort_flag) AS low_medium_confidence_rows,
    SUM(score_effort_misalignment_flag) AS score_effort_misalignment_rows
FROM activity_labeled_v2
""").fetchdf())

print('AI effort source distribution')
display(con.execute("""
SELECT ai_effort_match_source, effort_level, ai_confidence, COUNT(*) AS rows
FROM activity_labeled_v2
GROUP BY 1,2,3
ORDER BY rows DESC
LIMIT 30
""").fetchdf())

print('DevZone contextual filepath override distribution')
display(con.execute("""
SELECT journey_signal, effort_level, COUNT(*) AS rows
FROM activity_labeled_v2
WHERE activity = 'devzone downloads'
GROUP BY 1,2
ORDER BY rows DESC
""").fetchdf())

print('Dictionary')
display(con.execute("""
SELECT *
FROM activity_dictionary_v2
ORDER BY fallback_journey_signal, fallback_effort_level, activity
""").fetchdf())

print('Event-level label distribution')
display(con.execute("""
SELECT journey_signal, effort_level, persona_hint, modality, COUNT(*) AS rows
FROM activity_labeled_v2
GROUP BY 1,2,3,4
ORDER BY rows DESC
LIMIT 30
""").fetchdf())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dictionary size and coverage


,dictionary_rows,distinct_activities,missing_labels
0,19,19,0.0


AI effort mapping coverage


,activity_rows,rows_with_ai_effort,pct_rows_with_ai_effort,low_medium_confidence_rows,score_effort_misalignment_rows
0,69347501,60022492.0,86.55,8329367.0,14127179.0


...

,journey_signal,effort_level,rows
0,Build,moderate,28616032
1,Evaluate,moderate,8476963
2,Discover,low,1128886
3,Evaluate,low,550248
4,Build,low,363229
5,Discover,moderate,336081
6,Evaluate,high,197348
7,Build,passive,34256
8,Build,high,32851
9,Discover,passive,5584


Dictionary


,activity,fallback_journey_signal,fallback_effort_level,fallback_effort_rank,modality
0,brev,Build,high,3.0,Cloud Workspace
1,devzone downloads,Build,high,3.0,Download
2,ngc downloads,Build,high,3.0,Download
3,forum contributions,Champion,high,3.0,Community
4,conference,Discover,moderate,2.0,Event
5,contests,Discover,moderate,2.0,Content
6,other events,Discover,moderate,2.0,Event
7,dev program membership,Discover,passive,0.0,Content
8,event registrations,Discover,passive,0.0,Event
9,hackathons,Discover,passive,0.0,Content


Event-level label distribution


,journey_signal,effort_level,persona_hint,modality,rows
0,Build,moderate,CUDA,Download,15752985
1,Build,moderate,Robotics,Download,7731906
2,Discover,passive,Other,Content,6322911
3,Discover,high,Other,Content,5857343
4,Build,moderate,Other,Download,4886100
5,Evaluate,moderate,CUDA,Download,4326306
6,Evaluate,moderate,Other,Download,2201515
7,Build,high,GenAI,Download,2122235
8,Build,high,Other,Download,2104337
9,Evaluate,moderate,Robotics,Download,1780441


## 6. Developer universe and anchor date

In [9]:
con.execute("""
CREATE OR REPLACE TABLE developer_universe_v2 AS
SELECT DISTINCT developer_id FROM activity_base_v2 WHERE developer_id IS NOT NULL
UNION
SELECT DISTINCT developer_id FROM contact_one_row_v2 WHERE developer_id IS NOT NULL
""")

date_summary = con.execute("""
SELECT MIN(activity_date) AS min_activity_date, MAX(activity_date) AS max_activity_date
FROM activity_labeled_v2
""").fetchdf()
ANCHOR_DATE = date_summary.loc[0, "max_activity_date"]
print("ANCHOR_DATE:", ANCHOR_DATE)

display(con.execute("""
SELECT
    COUNT(*) AS universe_developers,
    SUM(CASE WHEN a.developer_id IS NOT NULL THEN 1 ELSE 0 END) AS developers_with_activity,
    SUM(CASE WHEN c.developer_id IS NOT NULL THEN 1 ELSE 0 END) AS developers_with_contact,
    SUM(CASE WHEN a.developer_id IS NOT NULL AND c.developer_id IS NULL THEN 1 ELSE 0 END) AS activity_without_contact,
    SUM(CASE WHEN a.developer_id IS NULL AND c.developer_id IS NOT NULL THEN 1 ELSE 0 END) AS contact_without_activity
FROM developer_universe_v2 u
LEFT JOIN (SELECT DISTINCT developer_id FROM activity_base_v2) a USING (developer_id)
LEFT JOIN contact_one_row_v2 c USING (developer_id)
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ANCHOR_DATE: 2026-03-12 00:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,universe_developers,developers_with_activity,developers_with_contact,activity_without_contact,contact_without_activity
0,9379190,7660278.0,8903197.0,475993.0,1718912.0


## 7. Non-overlapping recency window features

The old notebook used cumulative windows. This version creates incremental windows to reduce collinearity:

- `0_30d`
- `30_90d`
- `90_180d`

In [10]:
WINDOWS = [
    ("0_30d", 0, 30),
    ("30_90d", 30, 90),
    ("90_180d", 90, 180),
]

def build_window_features(label: str, start_days_ago: int, end_days_ago: int) -> None:
    table_name = f"dev_features_{label}_v2"
    con.execute(f"""
    CREATE OR REPLACE TABLE {table_name} AS
    WITH max_dt AS (SELECT MAX(activity_date) AS anchor_date FROM activity_labeled_v2),
    agg AS (
        SELECT
            developer_id,
            COUNT(*) AS activity_count,
            SUM(activity_score) AS activity_score_sum,
            AVG(activity_score) AS activity_score_avg,
            COUNT(DISTINCT activity_date) AS unique_activity_days,
            COUNT(DISTINCT activity) AS unique_activity_types,
            COUNT(DISTINCT modality) AS unique_modalities,
            COUNT(DISTINCT DATE_TRUNC('week', activity_date)) AS active_weeks,
            MIN(activity_date) AS first_activity_date_window,
            MAX(activity_date) AS last_activity_date_window,
            SUM(CASE WHEN journey_signal = 'Discover' THEN 1 ELSE 0 END) AS discover_count,
            SUM(CASE WHEN journey_signal = 'Learn' THEN 1 ELSE 0 END) AS learn_count,
            SUM(CASE WHEN journey_signal = 'Evaluate' THEN 1 ELSE 0 END) AS evaluate_count,
            SUM(CASE WHEN journey_signal = 'Build' THEN 1 ELSE 0 END) AS build_count,
            SUM(CASE WHEN journey_signal = 'Champion' THEN 1 ELSE 0 END) AS champion_count,
            SUM(CASE WHEN effort_rank >= 3 THEN 1 ELSE 0 END) AS high_effort_count,
            AVG(effort_rank) AS avg_effort_rank,
            MAX(effort_rank) AS max_effort_rank,
            SUM(confidence_weighted_effort_rank) AS total_confidence_weighted_effort,
            AVG(score_effort_gap) AS avg_score_effort_gap,
            SUM(score_effort_misalignment_flag) AS score_effort_misalignment_count,
            SUM(low_medium_confidence_effort_flag) AS low_medium_confidence_effort_count,
            SUM(effort_x_activity_score) AS effort_x_activity_score_sum,
            SUM(CASE WHEN modality = 'Download' THEN 1 ELSE 0 END) AS download_count,
            SUM(CASE WHEN modality = 'Hosted API' THEN 1 ELSE 0 END) AS hosted_api_count,
            SUM(CASE WHEN modality = 'Cloud Workspace' THEN 1 ELSE 0 END) AS cloud_workspace_count,
            SUM(CASE WHEN modality = 'Community' THEN 1 ELSE 0 END) AS community_count,
            SUM(CASE WHEN modality = 'Training' THEN 1 ELSE 0 END) AS training_count,
            SUM(CASE WHEN modality = 'Event' THEN 1 ELSE 0 END) AS event_count,
            SUM(cuda_persona_score) AS cuda_score,
            SUM(genai_persona_score) AS genai_score,
            SUM(robotics_persona_score) AS robotics_score,
            SUM(simulation_persona_score) AS simulation_score,
            SUM(learning_community_persona_score) AS learning_community_score
        FROM activity_labeled_v2, max_dt
        WHERE activity_date > anchor_date - INTERVAL {end_days_ago} DAY
          AND activity_date <= anchor_date - INTERVAL {start_days_ago} DAY
        GROUP BY developer_id
    )
    SELECT
        u.developer_id,
        COALESCE(a.activity_count, 0) AS activity_count,
        COALESCE(a.activity_score_sum, 0) AS activity_score_sum,
        COALESCE(a.activity_score_avg, 0) AS activity_score_avg,
        COALESCE(a.unique_activity_days, 0) AS unique_activity_days,
        COALESCE(a.unique_activity_types, 0) AS unique_activity_types,
        COALESCE(a.unique_modalities, 0) AS unique_modalities,
        COALESCE(a.active_weeks, 0) AS active_weeks,
        a.first_activity_date_window,
        a.last_activity_date_window,
        CASE WHEN a.last_activity_date_window IS NULL THEN 1 ELSE 0 END AS is_missing_last_activity_window,
        DATE_DIFF('day', a.last_activity_date_window, (SELECT anchor_date FROM max_dt)) AS days_since_last_activity_window,
        COALESCE(a.discover_count, 0) AS discover_count,
        COALESCE(a.learn_count, 0) AS learn_count,
        COALESCE(a.evaluate_count, 0) AS evaluate_count,
        COALESCE(a.build_count, 0) AS build_count,
        COALESCE(a.champion_count, 0) AS champion_count,
        COALESCE(a.high_effort_count, 0) AS high_effort_count,
        COALESCE(a.avg_effort_rank, 0) AS avg_effort_rank,
        COALESCE(a.max_effort_rank, 0) AS max_effort_rank,
        COALESCE(a.total_confidence_weighted_effort, 0) AS total_confidence_weighted_effort,
        COALESCE(a.avg_score_effort_gap, 0) AS avg_score_effort_gap,
        COALESCE(a.score_effort_misalignment_count, 0) AS score_effort_misalignment_count,
        COALESCE(a.low_medium_confidence_effort_count, 0) AS low_medium_confidence_effort_count,
        COALESCE(a.effort_x_activity_score_sum, 0) AS effort_x_activity_score_sum,
        COALESCE(a.download_count, 0) AS download_count,
        COALESCE(a.hosted_api_count, 0) AS hosted_api_count,
        COALESCE(a.cloud_workspace_count, 0) AS cloud_workspace_count,
        COALESCE(a.community_count, 0) AS community_count,
        COALESCE(a.training_count, 0) AS training_count,
        COALESCE(a.event_count, 0) AS event_count,
        COALESCE(a.cuda_score, 0) AS cuda_score,
        COALESCE(a.genai_score, 0) AS genai_score,
        COALESCE(a.robotics_score, 0) AS robotics_score,
        COALESCE(a.simulation_score, 0) AS simulation_score,
        COALESCE(a.learning_community_score, 0) AS learning_community_score,
        CASE WHEN COALESCE(a.activity_count, 0) > 0 THEN 1 ELSE 0 END AS has_activity,
        LN(1 + COALESCE(a.activity_count, 0)) AS log_activity_count,
        LN(1 + COALESCE(a.activity_score_sum, 0)) AS log_activity_score_sum,
        LN(1 + COALESCE(a.build_count, 0)) AS log_build_count,
        LN(1 + COALESCE(a.high_effort_count, 0)) AS log_high_effort_count,
        COALESCE(a.activity_count, 0) * 1.0 / NULLIF(COALESCE(a.unique_activity_days, 0), 0) AS activity_per_active_day,
        COALESCE(a.build_count, 0) * 1.0 / NULLIF(COALESCE(a.activity_count, 0), 0) AS build_share,
        COALESCE(a.high_effort_count, 0) * 1.0 / NULLIF(COALESCE(a.activity_count, 0), 0) AS high_effort_share
    FROM developer_universe_v2 u
    LEFT JOIN agg a USING (developer_id)
    """)

for label, start, end in WINDOWS:
    build_window_features(label, start, end)
    print(f"Built dev_features_{label}_v2")
    display(con.execute(f"""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT developer_id) AS developers,
           AVG(has_activity) AS pct_with_activity,
           MAX(activity_count) AS max_activity_count
    FROM dev_features_{label}_v2
    """).fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Built dev_features_0_30d_v2


,rows,developers,pct_with_activity,max_activity_count
0,9379190,9379190,0.044572,144228


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Built dev_features_30_90d_v2


,rows,developers,pct_with_activity,max_activity_count
0,9379190,9379190,0.045342,272221


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Built dev_features_90_180d_v2


,rows,developers,pct_with_activity,max_activity_count
0,9379190,9379190,0.055522,354569


## 8. Combine recency windows into one developer-level table

In [11]:
con.execute("""
CREATE OR REPLACE TABLE dev_recency_features_v2 AS
SELECT
    u.developer_id,

    f0.activity_count AS activity_count_0_30d,
    f1.activity_count AS activity_count_30_90d,
    f2.activity_count AS activity_count_90_180d,
    f0.has_activity AS has_activity_0_30d,
    f1.has_activity AS has_activity_30_90d,
    f2.has_activity AS has_activity_90_180d,
    f0.log_activity_count AS log_activity_count_0_30d,
    f1.log_activity_count AS log_activity_count_30_90d,
    f2.log_activity_count AS log_activity_count_90_180d,

    f0.build_count AS build_count_0_30d,
    f1.build_count AS build_count_30_90d,
    f2.build_count AS build_count_90_180d,
    f0.log_build_count AS log_build_count_0_30d,
    f1.log_build_count AS log_build_count_30_90d,
    f2.log_build_count AS log_build_count_90_180d,

    f0.high_effort_count AS high_effort_count_0_30d,
    f1.high_effort_count AS high_effort_count_30_90d,
    f2.high_effort_count AS high_effort_count_90_180d,

    f0.unique_activity_days AS unique_activity_days_0_30d,
    f0.unique_activity_types AS unique_activity_types_0_30d,
    f0.unique_modalities AS unique_modalities_0_30d,
    f0.activity_per_active_day AS activity_per_active_day_0_30d,
    f0.build_share AS build_share_0_30d,
    f0.high_effort_share AS high_effort_share_0_30d,

    f0.avg_effort_rank AS avg_effort_rank_0_30d,
    f1.avg_effort_rank AS avg_effort_rank_30_90d,
    f2.avg_effort_rank AS avg_effort_rank_90_180d,

    f0.total_confidence_weighted_effort AS total_confidence_weighted_effort_0_30d,
    f1.total_confidence_weighted_effort AS total_confidence_weighted_effort_30_90d,
    f2.total_confidence_weighted_effort AS total_confidence_weighted_effort_90_180d,

    -- FIXED: compute instead of referencing missing column
    (f0.total_confidence_weighted_effort * 1.0 / NULLIF(f0.activity_count, 0)) 
        AS confidence_weighted_effort_per_activity_0_30d,

    f0.avg_score_effort_gap AS avg_score_effort_gap_0_30d,
    f0.score_effort_misalignment_count AS score_effort_misalignment_count_0_30d,

    -- FIXED: compute instead of referencing missing column
    (f0.score_effort_misalignment_count * 1.0 / NULLIF(f0.activity_count, 0)) 
        AS score_effort_misalignment_share_0_30d,

    f0.low_medium_confidence_effort_count AS low_medium_confidence_effort_count_0_30d,
    f0.effort_x_activity_score_sum AS effort_x_activity_score_sum_0_30d,

    f0.days_since_last_activity_window AS days_since_last_activity_0_30d,
    f0.is_missing_last_activity_window AS is_missing_last_activity_0_30d,

    -- Velocity and recency-decay features
    f0.activity_count * 1.0 / NULLIF(f1.activity_count, 0) AS activity_velocity_0_30_vs_30_90,
    f0.build_count * 1.0 / NULLIF(f1.build_count, 0) AS build_velocity_0_30_vs_30_90,

    (0.60 * f0.activity_count + 0.30 * f1.activity_count + 0.10 * f2.activity_count) 
        AS weighted_recent_activity,

    (0.60 * f0.build_count + 0.30 * f1.build_count + 0.10 * f2.build_count) 
        AS weighted_recent_build,

    (0.60 * f0.total_confidence_weighted_effort + 0.30 * f1.total_confidence_weighted_effort + 0.10 * f2.total_confidence_weighted_effort) 
        AS weighted_recent_confidence_effort,

    -- Interaction features
    CASE WHEN f0.activity_count > 0 AND f0.build_count = 0 THEN 1 ELSE 0 END 
        AS active_non_builder_0_30d,

    CASE WHEN f0.activity_count = 0 AND f1.activity_count > 0 THEN 1 ELSE 0 END 
        AS newly_inactive_0_30d,

    CASE WHEN f0.build_count > 0 AND f0.activity_count <= 2 THEN 1 ELSE 0 END 
        AS low_volume_builder_0_30d,

    CASE WHEN f0.high_effort_count > 0 THEN 1 ELSE 0 END 
        AS has_high_effort_0_30d,

    CASE WHEN f0.build_count > 0 OR f0.hosted_api_count > 0 OR f0.cloud_workspace_count > 0 THEN 1 ELSE 0 END 
        AS recent_build_flag,

    CASE WHEN f0.champion_count > 0 THEN 1 ELSE 0 END 
        AS recent_champion_flag

FROM developer_universe_v2 u
LEFT JOIN dev_features_0_30d_v2 f0 USING (developer_id)
LEFT JOIN dev_features_30_90d_v2 f1 USING (developer_id)
LEFT JOIN dev_features_90_180d_v2 f2 USING (developer_id);""")

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    AVG(has_activity_0_30d) AS pct_active_0_30d,
    AVG(newly_inactive_0_30d) AS pct_newly_inactive_0_30d
FROM dev_recency_features_v2
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,developers,pct_active_0_30d,pct_newly_inactive_0_30d
0,9379190,9379190,0.044572,0.03801


## 9. Lifetime features with normalization, log transforms, and clipping helpers

Update: Lifetime features now include lifecycle classification columns from the colleague notebook: `user_type`, `max_stage_reached`, and `lifetime_devzone_download_count`.


In [12]:
con.execute("""
CREATE OR REPLACE TABLE dev_features_lifetime_v2 AS
WITH agg AS (
    SELECT
        developer_id,
        COUNT(*) AS lifetime_activity_count,
        SUM(activity_score) AS lifetime_activity_score_sum,
        AVG(activity_score) AS lifetime_activity_score_avg,
        COUNT(DISTINCT activity_date) AS lifetime_unique_activity_days,
        COUNT(DISTINCT activity) AS lifetime_unique_activity_types,
        COUNT(DISTINCT modality) AS lifetime_unique_modalities,
        COUNT(DISTINCT DATE_TRUNC('week', activity_date)) AS lifetime_active_weeks,
        MIN(activity_date) AS lifetime_first_activity_date,
        MAX(activity_date) AS lifetime_last_activity_date,
        SUM(CASE WHEN journey_signal = 'Discover' THEN 1 ELSE 0 END) AS lifetime_discover_count,
        SUM(CASE WHEN journey_signal = 'Learn' THEN 1 ELSE 0 END) AS lifetime_learn_count,
        SUM(CASE WHEN journey_signal = 'Evaluate' THEN 1 ELSE 0 END) AS lifetime_evaluate_count,
        SUM(CASE WHEN journey_signal = 'Build' THEN 1 ELSE 0 END) AS lifetime_build_count,
        SUM(CASE WHEN journey_signal = 'Champion' THEN 1 ELSE 0 END) AS lifetime_champion_count,
        SUM(CASE WHEN effort_rank >= 3 THEN 1 ELSE 0 END) AS lifetime_high_effort_count,
        AVG(effort_rank) AS lifetime_avg_effort_rank,
        MAX(effort_rank) AS lifetime_max_effort_rank,
        SUM(confidence_weighted_effort_rank) AS lifetime_total_confidence_weighted_effort,
        AVG(score_effort_gap) AS lifetime_avg_score_effort_gap,
        SUM(score_effort_misalignment_flag) AS lifetime_score_effort_misalignment_count,
        SUM(low_medium_confidence_effort_flag) AS lifetime_low_medium_confidence_effort_count,
        SUM(effort_x_activity_score) AS lifetime_effort_x_activity_score_sum,
        SUM(CASE WHEN activity = 'dli training' THEN 1 ELSE 0 END) AS lifetime_dli_training_count,
        SUM(CASE WHEN activity = 'webinars' THEN 1 ELSE 0 END) AS lifetime_webinar_count,
        SUM(CASE WHEN activity = 'forum contributions' THEN 1 ELSE 0 END) AS lifetime_forum_count,
        SUM(CASE WHEN activity = 'bugs filed' THEN 1 ELSE 0 END) AS lifetime_bug_count,
        SUM(CASE WHEN activity = 'hackathons' THEN 1 ELSE 0 END) AS lifetime_hackathon_count,
        SUM(CASE WHEN activity = 'model api' OR activity = 'hosted api' THEN 1 ELSE 0 END) AS lifetime_api_count,
        SUM(CASE WHEN activity = 'devzone downloads' THEN 1 ELSE 0 END) AS lifetime_devzone_download_count,
        SUM(CASE WHEN activity = 'ngc downloads' THEN 1 ELSE 0 END) AS lifetime_ngc_download_count,
        SUM(cuda_persona_score) AS cuda_score,
        SUM(genai_persona_score) AS genai_score,
        SUM(robotics_persona_score) AS robotics_score,
        SUM(simulation_persona_score) AS simulation_score,
        SUM(learning_community_persona_score) AS learning_community_score,
        SUM(CASE WHEN persona_hint = 'Other' THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END) AS other_persona_score
    FROM activity_labeled_v2
    GROUP BY developer_id
),
filled AS (
    SELECT
        u.developer_id,
        COALESCE(a.lifetime_activity_count, 0) AS lifetime_activity_count,
        COALESCE(a.lifetime_activity_score_sum, 0) AS lifetime_activity_score_sum,
        COALESCE(a.lifetime_activity_score_avg, 0) AS lifetime_activity_score_avg,
        COALESCE(a.lifetime_unique_activity_days, 0) AS lifetime_unique_activity_days,
        COALESCE(a.lifetime_unique_activity_types, 0) AS lifetime_unique_activity_types,
        COALESCE(a.lifetime_unique_modalities, 0) AS lifetime_unique_modalities,
        COALESCE(a.lifetime_active_weeks, 0) AS lifetime_active_weeks,
        a.lifetime_first_activity_date,
        a.lifetime_last_activity_date,
        COALESCE(a.lifetime_discover_count, 0) AS lifetime_discover_count,
        COALESCE(a.lifetime_learn_count, 0) AS lifetime_learn_count,
        COALESCE(a.lifetime_evaluate_count, 0) AS lifetime_evaluate_count,
        COALESCE(a.lifetime_build_count, 0) AS lifetime_build_count,
        COALESCE(a.lifetime_champion_count, 0) AS lifetime_champion_count,
        COALESCE(a.lifetime_high_effort_count, 0) AS lifetime_high_effort_count,
        COALESCE(a.lifetime_avg_effort_rank, 0) AS lifetime_avg_effort_rank,
        COALESCE(a.lifetime_max_effort_rank, 0) AS lifetime_max_effort_rank,
        COALESCE(a.lifetime_total_confidence_weighted_effort, 0) AS lifetime_total_confidence_weighted_effort,
        COALESCE(a.lifetime_avg_score_effort_gap, 0) AS lifetime_avg_score_effort_gap,
        COALESCE(a.lifetime_score_effort_misalignment_count, 0) AS lifetime_score_effort_misalignment_count,
        COALESCE(a.lifetime_low_medium_confidence_effort_count, 0) AS lifetime_low_medium_confidence_effort_count,
        COALESCE(a.lifetime_effort_x_activity_score_sum, 0) AS lifetime_effort_x_activity_score_sum,
        COALESCE(a.lifetime_dli_training_count, 0) AS lifetime_dli_training_count,
        COALESCE(a.lifetime_webinar_count, 0) AS lifetime_webinar_count,
        COALESCE(a.lifetime_forum_count, 0) AS lifetime_forum_count,
        COALESCE(a.lifetime_bug_count, 0) AS lifetime_bug_count,
        COALESCE(a.lifetime_hackathon_count, 0) AS lifetime_hackathon_count,
        COALESCE(a.lifetime_api_count, 0) AS lifetime_api_count,
        COALESCE(a.lifetime_devzone_download_count, 0) AS lifetime_devzone_download_count,
        COALESCE(a.lifetime_ngc_download_count, 0) AS lifetime_ngc_download_count,
        COALESCE(a.cuda_score, 0) AS cuda_score,
        COALESCE(a.genai_score, 0) AS genai_score,
        COALESCE(a.robotics_score, 0) AS robotics_score,
        COALESCE(a.simulation_score, 0) AS simulation_score,
        COALESCE(a.learning_community_score, 0) AS learning_community_score,
        COALESCE(a.other_persona_score, 0) AS other_persona_score
    FROM developer_universe_v2 u
    LEFT JOIN agg a USING (developer_id)
),
p99 AS (
    SELECT
        APPROX_QUANTILE(lifetime_activity_count, 0.99) AS p99_activity_count,
        APPROX_QUANTILE(lifetime_build_count, 0.99) AS p99_build_count,
        APPROX_QUANTILE(lifetime_high_effort_count, 0.99) AS p99_high_effort_count,
        APPROX_QUANTILE(lifetime_total_confidence_weighted_effort, 0.99) AS p99_total_confidence_weighted_effort,
        APPROX_QUANTILE(lifetime_effort_x_activity_score_sum, 0.99) AS p99_effort_x_activity_score_sum,
        APPROX_QUANTILE(lifetime_activity_score_sum, 0.99) AS p99_activity_score_sum
    FROM filled
)
SELECT
    f.*,
    CASE
        WHEN f.lifetime_unique_activity_days = 1 THEN 'tourist'
        WHEN f.lifetime_build_count + f.lifetime_champion_count <= 2
          AND f.lifetime_high_effort_count = 0
          AND f.lifetime_devzone_download_count >= 1
        THEN 'free_email_user'
        ELSE 'real_user'
    END AS user_type,
    CASE
        WHEN f.lifetime_champion_count >= 1 THEN 'Champion'
        WHEN f.lifetime_build_count >= 1 THEN 'Build'
        WHEN f.lifetime_evaluate_count >= 1 THEN 'Evaluate'
        WHEN f.lifetime_learn_count >= 1 THEN 'Learn'
        WHEN f.lifetime_discover_count >= 1 THEN 'Discover'
        ELSE 'None'
    END AS max_stage_reached,
    CASE WHEN lifetime_activity_count > 0 THEN 1 ELSE 0 END AS has_lifetime_activity,
    LN(1 + lifetime_activity_count) AS log_lifetime_activity_count,
    LN(1 + lifetime_activity_score_sum) AS log_lifetime_activity_score_sum,
    LN(1 + lifetime_build_count) AS log_lifetime_build_count,
    LN(1 + lifetime_high_effort_count) AS log_lifetime_high_effort_count,
    lifetime_total_confidence_weighted_effort * 1.0 / NULLIF(lifetime_activity_count, 0) AS effort_per_activity_lifetime,
    lifetime_score_effort_misalignment_count * 1.0 / NULLIF(lifetime_activity_count, 0) AS score_effort_misalignment_share_lifetime,
    LN(1 + lifetime_total_confidence_weighted_effort) AS log_lifetime_total_confidence_weighted_effort,
    LN(1 + lifetime_effort_x_activity_score_sum) AS log_lifetime_effort_x_activity_score_sum,
    LEAST(lifetime_activity_count, p99.p99_activity_count) AS clipped_lifetime_activity_count_p99,
    LEAST(lifetime_build_count, p99.p99_build_count) AS clipped_lifetime_build_count_p99,
    LEAST(lifetime_activity_score_sum, p99.p99_activity_score_sum) AS clipped_lifetime_activity_score_sum_p99,
    LEAST(lifetime_total_confidence_weighted_effort, p99.p99_total_confidence_weighted_effort) AS clipped_lifetime_total_confidence_weighted_effort_p99,
    LEAST(lifetime_effort_x_activity_score_sum, p99.p99_effort_x_activity_score_sum) AS clipped_lifetime_effort_x_activity_score_sum_p99,
    LN(1 + LEAST(lifetime_activity_count, p99.p99_activity_count)) AS log_clipped_lifetime_activity_count_p99,
    LN(1 + LEAST(lifetime_activity_score_sum, p99.p99_activity_score_sum)) AS log_clipped_lifetime_activity_score_sum_p99,
    LN(1 + LEAST(lifetime_total_confidence_weighted_effort, p99.p99_total_confidence_weighted_effort)) AS log_clipped_lifetime_total_confidence_weighted_effort_p99,
    LN(1 + LEAST(lifetime_effort_x_activity_score_sum, p99.p99_effort_x_activity_score_sum)) AS log_clipped_lifetime_effort_x_activity_score_sum_p99,
    lifetime_activity_count * 1.0 / NULLIF(lifetime_active_weeks, 0) AS activity_per_active_week_lifetime,
    lifetime_build_count * 1.0 / NULLIF(lifetime_activity_count, 0) AS build_share_lifetime,
    lifetime_high_effort_count * 1.0 / NULLIF(lifetime_activity_count, 0) AS high_effort_share_lifetime
FROM filled f
CROSS JOIN p99
""")

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    MAX(lifetime_activity_count) AS max_raw_activity_count,
    MAX(clipped_lifetime_activity_count_p99) AS max_clipped_activity_count
FROM dev_features_lifetime_v2
""").fetchdf())

print('New lifecycle feature distributions')
display(con.execute("""
SELECT user_type, max_stage_reached, COUNT(*) AS developers
FROM dev_features_lifetime_v2
GROUP BY 1,2
ORDER BY developers DESC
LIMIT 30
""").fetchdf())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,developers,max_raw_activity_count,max_clipped_activity_count
0,9379190,9379190,2027375,116


New lifecycle feature distributions


,user_type,max_stage_reached,developers
0,tourist,Discover,1902772
1,real_user,None,1718912
2,tourist,Build,1448366
3,real_user,Build,1425799
4,tourist,Learn,895278
5,tourist,Evaluate,718179
6,real_user,Learn,554489
7,real_user,Evaluate,284132
8,real_user,Discover,238602
9,real_user,Champion,71189


## 10. Activation and dormancy status

This block must run **after** `activity_labeled_v2` and `dev_features_lifetime_v2` are created.

It creates:

- `dev_weekly_features_v2`
- `dev_meaningful_week_v2`
- `dev_activation_v2`
- `dev_dormancy_status_v2`

This fixes the dependency issue where `dev_dormancy_status_v2` was referenced before being created.

In [13]:
ACTIVE_CUTOFF_DAYS = 30
COOLING_CUTOFF_DAYS = 90
DORMANT_CUTOFF_DAYS = 365

# Weekly activity summary used to identify meaningful weeks.
con.execute("""
CREATE OR REPLACE TABLE dev_weekly_features_v2 AS
SELECT
    developer_id,
    DATE_TRUNC('week', activity_date) AS week_start,
    COUNT(*) AS activity_count_total,
    SUM(activity_score) AS activity_score_sum,
    COUNT(DISTINCT activity) AS unique_activity_types,
    SUM(CASE WHEN journey_signal = 'Build' THEN 1 ELSE 0 END) AS build_count,
    SUM(CASE WHEN journey_signal = 'Champion' THEN 1 ELSE 0 END) AS champion_count,
    SUM(CASE WHEN effort_level = 'High' THEN 1 ELSE 0 END) AS high_effort_count,
    SUM(CASE WHEN modality IN ('Hosted API', 'Cloud Workspace') THEN 1 ELSE 0 END) AS product_use_count
FROM activity_labeled_v2
GROUP BY 1, 2
""")

# Meaningful week = a week with clear product use, build/champion activity, high-effort activity,
# or enough repeated activity to indicate more than a one-off touch.
con.execute("""
CREATE OR REPLACE TABLE dev_meaningful_week_v2 AS
SELECT
    *,
    CASE
        WHEN build_count > 0
          OR champion_count > 0
          OR high_effort_count > 0
          OR product_use_count > 0
          OR activity_count_total >= 3
        THEN 1 ELSE 0
    END AS meaningful_week_flag
FROM dev_weekly_features_v2
""")

# Activation is based on any lifetime activity. Meaningful weeks are kept as supporting context.
con.execute("""
CREATE OR REPLACE TABLE dev_activation_v2 AS
WITH meaningful AS (
    SELECT
        developer_id,
        COALESCE(SUM(meaningful_week_flag), 0) AS lifetime_meaningful_weeks,
        MIN(CASE WHEN meaningful_week_flag = 1 THEN week_start END) AS first_meaningful_week_start,
        MAX(CASE WHEN meaningful_week_flag = 1 THEN week_start END) AS last_meaningful_week_start
    FROM dev_meaningful_week_v2
    GROUP BY developer_id
)
SELECT
    u.developer_id,
    CASE WHEN COALESCE(l.lifetime_activity_count, 0) > 0 THEN 1 ELSE 0 END AS is_activated,
    COALESCE(l.lifetime_activity_count, 0) AS lifetime_activity_count_for_activation,
    COALESCE(m.lifetime_meaningful_weeks, 0) AS lifetime_meaningful_weeks,
    m.first_meaningful_week_start,
    m.last_meaningful_week_start,
    l.lifetime_first_activity_date,
    l.lifetime_last_activity_date
FROM developer_universe_v2 u
LEFT JOIN dev_features_lifetime_v2 l USING (developer_id)
LEFT JOIN meaningful m USING (developer_id)
""")

# Dormancy is a lifecycle / recency state, separate from behavior journey stage.
con.execute(f"""
CREATE OR REPLACE TABLE dev_dormancy_status_v2 AS
WITH max_dt AS (
    SELECT MAX(activity_date) AS anchor_date
    FROM activity_labeled_v2
),
base AS (
    SELECT
        a.*,
        CASE
            WHEN a.lifetime_last_activity_date IS NOT NULL THEN
                DATE_DIFF('day', a.lifetime_last_activity_date, (SELECT anchor_date FROM max_dt))
            ELSE NULL
        END AS days_since_last_activity,
        CASE
            WHEN a.last_meaningful_week_start IS NOT NULL THEN
                DATE_DIFF('day', a.last_meaningful_week_start, (SELECT anchor_date FROM max_dt))
            ELSE NULL
        END AS days_since_last_meaningful_week
    FROM dev_activation_v2 a
)
SELECT
    *,
    CASE
        WHEN lifetime_activity_count_for_activation = 0 THEN 'Unactivated'
        WHEN days_since_last_activity < {ACTIVE_CUTOFF_DAYS} THEN 'Active'
        WHEN days_since_last_activity < {COOLING_CUTOFF_DAYS} THEN 'Cooling'
        WHEN days_since_last_activity < {DORMANT_CUTOFF_DAYS} THEN 'At_Risk'
        ELSE 'Dormant'
    END AS dormancy_status,
    CASE
        WHEN lifetime_activity_count_for_activation > 0
         AND days_since_last_activity >= {DORMANT_CUTOFF_DAYS}
        THEN 1 ELSE 0
    END AS dormant_flag,
    CASE
        WHEN lifetime_activity_count_for_activation > 0
         AND days_since_last_activity >= {COOLING_CUTOFF_DAYS}
         AND days_since_last_activity < {DORMANT_CUTOFF_DAYS}
        THEN 1 ELSE 0
    END AS at_risk_flag,
    CASE
        WHEN lifetime_activity_count_for_activation > 0
         AND days_since_last_activity >= {ACTIVE_CUTOFF_DAYS}
         AND days_since_last_activity < {COOLING_CUTOFF_DAYS}
        THEN 1 ELSE 0
    END AS cooling_flag
FROM base
""")

print("Dormancy distribution")
display(con.execute("""
SELECT
    dormancy_status,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct,
    AVG(days_since_last_activity) AS avg_days_since_last_activity,
    MEDIAN(days_since_last_activity) AS median_days_since_last_activity
FROM dev_dormancy_status_v2
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dormancy distribution


,dormancy_status,developers,pct,avg_days_since_last_activity,median_days_since_last_activity
0,Dormant,5304852,56.56,1103.341224,1037.0
1,Unactivated,1718912,18.33,NaN,NaN
2,At_Risk,1580877,16.86,237.897645,240.0
3,Active,418049,4.46,11.338460,10.0
4,Cooling,356500,3.80,55.597027,54.0


## 10. Developer-level effort score and effort bands

This replaces raw `activity_score` as the main effort segmentation feature. It combines capped lifetime volume, capped lifetime score, activity breadth, distinct assets, recent activity, and high-signal activity types. The caps and `LN(1+x)` transforms keep extreme repeated downloads from dominating the segmentation.


In [14]:

con.execute("""
CREATE OR REPLACE TABLE dev_effort_level_v2 AS
WITH base AS (
    SELECT
        lf.developer_id,
        lf.lifetime_activity_count,
        lf.lifetime_activity_score_sum,
        lf.lifetime_unique_activity_types,
        lf.lifetime_unique_modalities,
        lf.lifetime_high_effort_count,
        lf.lifetime_avg_effort_rank,
        lf.lifetime_max_effort_rank,
        lf.lifetime_total_confidence_weighted_effort,
        lf.lifetime_avg_score_effort_gap,
        lf.lifetime_score_effort_misalignment_count,
        lf.lifetime_low_medium_confidence_effort_count,
        lf.lifetime_effort_x_activity_score_sum,
        lf.lifetime_build_count,
        lf.lifetime_champion_count,
        lf.lifetime_dli_training_count,
        lf.lifetime_webinar_count,
        lf.lifetime_forum_count,
        lf.lifetime_bug_count,
        lf.lifetime_hackathon_count,
        lf.lifetime_api_count,
        lf.lifetime_devzone_download_count,
        lf.lifetime_ngc_download_count,
        lf.log_clipped_lifetime_activity_count_p99,
        lf.log_clipped_lifetime_activity_score_sum_p99,
        lf.log_clipped_lifetime_total_confidence_weighted_effort_p99,
        lf.log_clipped_lifetime_effort_x_activity_score_sum_p99,
        COALESCE(r.activity_count_0_30d, 0) AS activity_count_0_30d,
        COALESCE(r.activity_count_30_90d, 0) AS activity_count_30_90d,
        COALESCE(r.activity_count_90_180d, 0) AS activity_count_90_180d,
        COALESCE(r.unique_activity_types_0_30d, 0) AS unique_activity_types_0_30d,
        COALESCE(r.weighted_recent_confidence_effort, 0) AS weighted_recent_confidence_effort,
        COALESCE(r.avg_effort_rank_0_30d, 0) AS avg_effort_rank_0_30d,
        COALESCE(r.recent_build_flag, 0) AS recent_build_flag,
        CASE
            WHEN COALESCE(r.activity_count_0_30d, 0) > 0 THEN 1.00
            WHEN COALESCE(r.activity_count_30_90d, 0) > 0 THEN 0.75
            WHEN COALESCE(r.activity_count_90_180d, 0) > 0 THEN 0.50
            ELSE 0.25
        END AS recency_weight
    FROM dev_features_lifetime_v2 lf
    LEFT JOIN dev_recency_features_v2 r USING (developer_id)
), scored AS (
    SELECT
        *,
        (
            -- Effort is now mostly driven by the AI-derived effort axis, with activity_score kept as a secondary business-value signal.
            0.30 * COALESCE(log_clipped_lifetime_total_confidence_weighted_effort_p99, 0)
          + 0.20 * COALESCE(log_clipped_lifetime_effort_x_activity_score_sum_p99, 0)
          + 0.15 * COALESCE(log_clipped_lifetime_activity_count_p99, 0)
          + 0.10 * LN(1 + COALESCE(lifetime_unique_activity_types, 0))
          + 0.10 * LN(1 + COALESCE(lifetime_unique_modalities, 0))
          + 0.10 * LN(1 + COALESCE(weighted_recent_confidence_effort, 0))
          + 0.05 * COALESCE(lifetime_avg_effort_rank, 0)
        ) * recency_weight AS developer_effort_score
    FROM base
), cutoffs AS (
    SELECT
        QUANTILE_CONT(developer_effort_score, 0.50) AS p50_effort,
        QUANTILE_CONT(developer_effort_score, 0.75) AS p75_effort,
        QUANTILE_CONT(developer_effort_score, 0.90) AS p90_effort
    FROM scored
    WHERE lifetime_activity_count > 0
)
SELECT
    s.*,
    CASE
        WHEN s.lifetime_activity_count = 0 THEN 'no activity'
        WHEN s.developer_effort_score >= c.p90_effort THEN 'very high effort'
        WHEN s.developer_effort_score >= c.p75_effort THEN 'high effort'
        WHEN s.developer_effort_score >= c.p50_effort THEN 'medium effort'
        ELSE 'low effort'
    END AS developer_effort_level,
    CASE
        WHEN s.lifetime_activity_count = 0 THEN 0
        WHEN s.developer_effort_score >= c.p90_effort THEN 4
        WHEN s.developer_effort_score >= c.p75_effort THEN 3
        WHEN s.developer_effort_score >= c.p50_effort THEN 2
        ELSE 1
    END AS developer_effort_rank
FROM scored s
CROSS JOIN cutoffs c
""")

print('Developer effort level distribution')
display(con.execute("""
SELECT
    developer_effort_level,
    developer_effort_rank,
    COUNT(*) AS developers,
    ROUND(AVG(developer_effort_score), 3) AS avg_effort_score,
    ROUND(AVG(lifetime_activity_count), 2) AS avg_lifetime_activity_count,
    ROUND(AVG(lifetime_activity_score_sum), 2) AS avg_lifetime_activity_score_sum,
    ROUND(AVG(lifetime_total_confidence_weighted_effort), 2) AS avg_lifetime_confidence_weighted_effort,
    ROUND(AVG(lifetime_avg_score_effort_gap), 2) AS avg_score_effort_gap,
    ROUND(AVG(lifetime_unique_activity_types), 2) AS avg_lifetime_activity_types
FROM dev_effort_level_v2
GROUP BY 1,2
ORDER BY developer_effort_rank DESC
""").fetchdf())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Developer effort level distribution


,developer_effort_level,developer_effort_rank,developers,avg_effort_score,avg_lifetime_activity_count,avg_lifetime_activity_score_sum,avg_lifetime_confidence_weighted_effort,avg_score_effort_gap,avg_lifetime_activity_types
0,very high effort,4,766034,1.462,42.89,137.27,79.61,-49.21,2.21
1,high effort,3,1151292,0.716,19.25,73.98,35.47,-48.93,2.50
2,medium effort,2,2152014,0.468,3.62,16.28,7.10,-51.52,2.19
3,low effort,1,3590938,0.294,1.82,3.65,3.09,-52.84,1.58
4,no activity,0,1718912,0.000,0.00,0.00,0.00,0.00,0.00


## 10. Contact-profile persona hints, then final persona

Activity-based persona is still primary. Contact profile text is added only after developer-level aggregation so it enriches persona without changing activity row counts.

In [15]:
con.execute(r"""
CREATE OR REPLACE TABLE dev_contact_persona_v2 AS
WITH base AS (
    SELECT
        developer_id,
        LOWER(
            COALESCE(development_areas, '') || ' ' ||
            COALESCE(fields_of_interest, '') || ' ' ||
            COALESCE(industry_segment_vertical, '')
        ) AS profile_text
    FROM contact_one_row_v2
)
SELECT
    developer_id,
    CASE WHEN REGEXP_MATCHES(profile_text, 'cuda|gpu|accelerated|hpc|rapids|cudnn') THEN 1.0 ELSE 0.0 END AS cuda_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'genai|generative|llm|ai|machine learning|deep learning|inference') THEN 1.0 ELSE 0.0 END AS genai_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'robot|isaac|ros|jetson|autonomous|edge') THEN 1.0 ELSE 0.0 END AS robotics_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'simulation|omniverse|digital twin|graphics|render|rtx') THEN 1.0 ELSE 0.0 END AS simulation_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'training|education|student|academic|community|developer program') THEN 1.0 ELSE 0.0 END AS learning_community_profile_score
FROM base
""")

con.execute("""
CREATE OR REPLACE TABLE dev_persona_v2 AS
WITH base AS (
    SELECT
        lf.developer_id,
        lf.cuda_score + COALESCE(cp.cuda_profile_score, 0) AS cuda_score,
        lf.genai_score + COALESCE(cp.genai_profile_score, 0) AS genai_score,
        lf.robotics_score + COALESCE(cp.robotics_profile_score, 0) AS robotics_score,
        lf.simulation_score + COALESCE(cp.simulation_profile_score, 0) AS simulation_score,
        lf.learning_community_score + COALESCE(cp.learning_community_profile_score, 0) AS learning_community_score,
        lf.other_persona_score
    FROM dev_features_lifetime_v2 lf
    LEFT JOIN dev_contact_persona_v2 cp USING (developer_id)
),
norm AS (
    SELECT *,
        cuda_score + genai_score + robotics_score + simulation_score + learning_community_score AS specific_persona_score,
        cuda_score + genai_score + robotics_score + simulation_score + learning_community_score + other_persona_score AS total_persona_score
    FROM base
),
shares AS (
    SELECT *,
        COALESCE(cuda_score / NULLIF(specific_persona_score, 0), 0) AS cuda_share,
        COALESCE(genai_score / NULLIF(specific_persona_score, 0), 0) AS genai_share,
        COALESCE(robotics_score / NULLIF(specific_persona_score, 0), 0) AS robotics_share,
        COALESCE(simulation_score / NULLIF(specific_persona_score, 0), 0) AS simulation_share,
        COALESCE(learning_community_score / NULLIF(specific_persona_score, 0), 0) AS learning_community_share,
        COALESCE(other_persona_score / NULLIF(total_persona_score, 0), 0) AS other_share
    FROM norm
),
entropy AS (
    SELECT *,
        -1 * (
            CASE WHEN cuda_share > 0 THEN cuda_share * LN(cuda_share) ELSE 0 END +
            CASE WHEN genai_share > 0 THEN genai_share * LN(genai_share) ELSE 0 END +
            CASE WHEN robotics_share > 0 THEN robotics_share * LN(robotics_share) ELSE 0 END +
            CASE WHEN simulation_share > 0 THEN simulation_share * LN(simulation_share) ELSE 0 END +
            CASE WHEN learning_community_share > 0 THEN learning_community_share * LN(learning_community_share) ELSE 0 END
        ) / LN(5) AS persona_entropy
    FROM shares
),
long_scores AS (
    SELECT developer_id, 'CUDA' AS persona, cuda_share AS score FROM entropy
    UNION ALL SELECT developer_id, 'GenAI', genai_share FROM entropy
    UNION ALL SELECT developer_id, 'Robotics', robotics_share FROM entropy
    UNION ALL SELECT developer_id, 'Simulation', simulation_share FROM entropy
    UNION ALL SELECT developer_id, 'Learning_Community', learning_community_share FROM entropy
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY developer_id ORDER BY score DESC, persona) AS rn,
        LEAD(score) OVER (PARTITION BY developer_id ORDER BY score DESC, persona) AS second_score
    FROM long_scores
)
SELECT
    e.*,
    CASE WHEN e.specific_persona_score = 0 THEN 'Unknown' ELSE r.persona END AS persona,
    CASE WHEN e.specific_persona_score = 0 THEN 0 ELSE r.score END AS persona_confidence,
    CASE
        WHEN e.specific_persona_score = 0 THEN 'Unknown'
        WHEN r.score >= 0.70 THEN 'High'
        WHEN r.score >= 0.45 THEN 'Medium'
        ELSE 'Low'
    END AS persona_confidence_tier,
    CASE
        WHEN e.specific_persona_score = 0 THEN 0
        WHEN e.persona_entropy >= 0.60 OR r.score - COALESCE(r.second_score, 0) <= 0.15 THEN 1
        ELSE 0
    END AS mixed_persona_flag
FROM entropy e
LEFT JOIN ranked r ON e.developer_id = r.developer_id AND r.rn = 1
""")

display(con.execute("""
SELECT persona, persona_confidence_tier, mixed_persona_flag, COUNT(*) AS developers
FROM dev_persona_v2
GROUP BY 1,2,3
ORDER BY developers DESC
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,persona,persona_confidence_tier,mixed_persona_flag,developers
0,Unknown,Unknown,0,1918828
1,CUDA,High,0,1834363
2,GenAI,High,0,1377082
3,CUDA,Medium,1,718196
4,CUDA,Medium,0,662938
5,GenAI,Medium,1,584715
6,Simulation,High,0,511271
7,Robotics,High,0,368202
8,CUDA,Low,1,317655
9,Learning_Community,High,0,314111


## 11. Journey state features from the recent window

For predictive modeling, avoid using journey-state labels if the target is derived from the same future/current activity window. Use the underlying count/rate features instead, or build journey state from a strictly prior window.

Dormancy / activation status
    ↓
Recent activity
    ↓
Intent signal
    ↓
Depth of behavior
    ↓
Trend
    ↓
Journey stage

## 12. Behavior journey stage and lifecycle overlay

This block separates behavior from lifecycle:

- `behavior_journey_stage_30d`: what the developer appears to do, based on behavior and intent.
- `dormancy_status`: lifecycle/recency state, created separately.
- `current_journey_state_30d`: optional combined view for reporting, not the primary modeling label.

This avoids making journey stage a simple relabeling of dormancy.


In [16]:
con.execute("""
CREATE OR REPLACE TABLE dev_journey_state_v2 AS
WITH base AS (
    SELECT
        r.developer_id,

        -- Recent activity
        COALESCE(r.activity_count_0_30d, 0) AS activity_count_0_30d,
        COALESCE(r.activity_count_30_90d, 0) AS activity_count_30_90d,
        COALESCE(r.activity_count_90_180d, 0) AS activity_count_90_180d,
        COALESCE(r.build_count_0_30d, 0) AS build_count_0_30d,
        COALESCE(r.build_count_30_90d, 0) AS build_count_30_90d,
        COALESCE(r.high_effort_count_0_30d, 0) AS high_effort_count_0_30d,
        COALESCE(r.high_effort_count_30_90d, 0) AS high_effort_count_30_90d,
        COALESCE(r.unique_activity_types_0_30d, 0) AS unique_activity_types_0_30d,
        COALESCE(r.unique_modalities_0_30d, 0) AS unique_modalities_0_30d,
        COALESCE(r.recent_build_flag, 0) AS recent_build_flag,
        COALESCE(r.recent_champion_flag, 0) AS recent_champion_flag,

        -- Lifecycle / dormancy
        COALESCE(d.is_activated, 0) AS is_activated,
        COALESCE(d.lifetime_activity_count_for_activation, 0) AS lifetime_activity_count_for_activation,
        COALESCE(d.dormancy_status, 'Unactivated') AS dormancy_status,
        CASE WHEN d.dormancy_status = 'Dormant' THEN 1 ELSE 0 END AS dormant_flag,
        CASE WHEN d.dormancy_status = 'At_Risk' THEN 1 ELSE 0 END AS at_risk_flag,
        CASE WHEN d.dormancy_status = 'Cooling' THEN 1 ELSE 0 END AS cooling_flag,
        d.days_since_last_activity,

        -- Lifetime context
        COALESCE(l.lifetime_activity_count, 0) AS lifetime_activity_count,
        COALESCE(l.lifetime_build_count, 0) AS lifetime_build_count,
        COALESCE(l.lifetime_champion_count, 0) AS lifetime_champion_count,
        COALESCE(l.lifetime_high_effort_count, 0) AS lifetime_high_effort_count,
        COALESCE(l.lifetime_unique_activity_types, 0) AS lifetime_unique_activity_types,
        COALESCE(l.lifetime_unique_modalities, 0) AS lifetime_unique_modalities,

        -- Trend only meaningful when there is activity in either comparison window.
        CASE
            WHEN COALESCE(r.activity_count_0_30d, 0) = 0 AND COALESCE(r.activity_count_30_90d, 0) = 0 THEN NULL
            WHEN COALESCE(r.activity_count_30_90d, 0) = 0 AND COALESCE(r.activity_count_0_30d, 0) > 0 THEN 2.0
            ELSE CAST(r.activity_count_0_30d AS DOUBLE) / NULLIF(CAST(r.activity_count_30_90d AS DOUBLE), 0)
        END AS recent_activity_trend_ratio

    FROM dev_recency_features_v2 r
    LEFT JOIN dev_dormancy_status_v2 d USING (developer_id)
    LEFT JOIN dev_features_lifetime_v2 l USING (developer_id)
),
scored AS (
    SELECT
        *,
        CASE
            WHEN activity_count_0_30d >= 5 THEN 'High'
            WHEN activity_count_0_30d >= 2 THEN 'Medium'
            WHEN activity_count_0_30d = 1 THEN 'Low'
            ELSE 'None'
        END AS activity_volume_band,

        -- Intent uses recent behavior first, then lifetime fallback for inactive historical users.
        CASE
            WHEN build_count_0_30d > 0 OR recent_build_flag = 1 OR lifetime_build_count >= 5 THEN 'Build_Intent'
            WHEN high_effort_count_0_30d > 0 OR lifetime_high_effort_count >= 5 THEN 'Evaluation_Intent'
            WHEN activity_count_0_30d > 0 THEN 'Learning_Intent'
            ELSE 'No_Recent_Intent'
        END AS intent_signal,

        CASE
            WHEN recent_activity_trend_ratio IS NULL THEN NULL
            WHEN recent_activity_trend_ratio >= 1.5 THEN 'Accelerating'
            WHEN recent_activity_trend_ratio >= 0.7 THEN 'Stable'
            WHEN recent_activity_trend_ratio > 0 THEN 'Declining'
            ELSE NULL
        END AS trend_signal
    FROM base
),
behavior AS (
    SELECT
        *,
        CASE
            WHEN lifetime_activity_count = 0 THEN 'Unactivated'
            WHEN build_count_0_30d > 0 OR recent_build_flag = 1 OR lifetime_build_count >= 10 THEN 'Builder'
            WHEN high_effort_count_0_30d > 0 OR lifetime_high_effort_count >= 10 THEN 'Evaluator'
            WHEN activity_count_0_30d > 0 AND unique_activity_types_0_30d >= 2 THEN 'Explorer'
            WHEN activity_count_0_30d > 0 THEN 'Learner'
            WHEN lifetime_activity_count > 0 THEN 'Historically_Active'
            ELSE 'Unactivated'
        END AS behavior_journey_stage_30d
    FROM scored
)
SELECT
    developer_id,
    behavior_journey_stage_30d,

    -- Optional combined reporting view. Use behavior_journey_stage_30d + dormancy_status for modeling.
    CASE
        WHEN behavior_journey_stage_30d = 'Unactivated' THEN 'Unactivated'
        WHEN dormancy_status IN ('Dormant', 'At_Risk', 'Cooling') THEN dormancy_status || '_' || behavior_journey_stage_30d
        ELSE behavior_journey_stage_30d
    END AS current_journey_state_30d,

    CASE
        WHEN behavior_journey_stage_30d = 'Unactivated' THEN 0
        WHEN behavior_journey_stage_30d = 'Historically_Active' THEN 1
        WHEN behavior_journey_stage_30d = 'Learner' THEN 2
        WHEN behavior_journey_stage_30d = 'Explorer' THEN 3
        WHEN behavior_journey_stage_30d = 'Evaluator' THEN 4
        WHEN behavior_journey_stage_30d = 'Builder' THEN 5
        ELSE 1
    END AS behavior_journey_rank_30d,

    -- Keep existing name for compatibility with downstream notebooks.
    CASE
        WHEN behavior_journey_stage_30d = 'Unactivated' THEN 0
        WHEN behavior_journey_stage_30d = 'Historically_Active' THEN 1
        WHEN behavior_journey_stage_30d = 'Learner' THEN 2
        WHEN behavior_journey_stage_30d = 'Explorer' THEN 3
        WHEN behavior_journey_stage_30d = 'Evaluator' THEN 4
        WHEN behavior_journey_stage_30d = 'Builder' THEN 5
        ELSE 1
    END AS current_journey_rank_30d,

    activity_volume_band,
    intent_signal,
    trend_signal,
    recent_activity_trend_ratio
FROM behavior
""")

print('Behavior journey stage distribution')
display(con.execute("""
SELECT
    behavior_journey_stage_30d,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM dev_journey_state_v2
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())

print('Combined journey state distribution')
display(con.execute("""
SELECT
    current_journey_state_30d,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM dev_journey_state_v2
GROUP BY 1
ORDER BY developers DESC
LIMIT 30
""").fetchdf())

print('Journey vs dormancy should no longer be one-to-one')
display(con.execute("""
SELECT
    js.behavior_journey_stage_30d,
    d.dormancy_status,
    COUNT(*) AS developers
FROM dev_journey_state_v2 js
LEFT JOIN dev_dormancy_status_v2 d USING (developer_id)
GROUP BY 1,2
ORDER BY developers DESC
LIMIT 50
""").fetchdf())

# Guardrails
print('Guardrail: Unactivated should mean no lifetime activity')
display(con.execute("""
SELECT COUNT(*) AS unactivated_with_lifetime_activity
FROM dev_journey_state_v2 j
JOIN dev_features_lifetime_v2 l USING (developer_id)
WHERE j.behavior_journey_stage_30d = 'Unactivated'
  AND l.lifetime_activity_count > 0
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Behavior journey stage distribution


,behavior_journey_stage_30d,developers,pct
0,Historically_Active,6963992,74.25
1,Unactivated,1718912,18.33
2,Builder,319991,3.41
3,Evaluator,289432,3.09
4,Learner,85487,0.91
5,Explorer,1376,0.01


Combined journey state distribution


,current_journey_state_30d,developers,pct
0,Dormant_Historically_Active,5132624,54.72
1,Unactivated,1718912,18.33
2,At_Risk_Historically_Active,1499923,15.99
3,Cooling_Historically_Active,331445,3.53
4,Evaluator,280897,2.99
5,Dormant_Builder,168052,1.79
6,Learner,85487,0.91
7,At_Risk_Builder,77223,0.82
8,Builder,50289,0.54
9,Cooling_Builder,24427,0.26


Journey vs dormancy should no longer be one-to-one


,behavior_journey_stage_30d,dormancy_status,developers
0,Historically_Active,Dormant,5132624
1,Unactivated,Unactivated,1718912
2,Historically_Active,At_Risk,1499923
3,Historically_Active,Cooling,331445
4,Evaluator,Active,280897
5,Builder,Dormant,168052
6,Learner,Active,85487
7,Builder,At_Risk,77223
8,Builder,Active,50289
9,Builder,Cooling,24427


Guardrail: Unactivated should mean no lifetime activity


,unactivated_with_lifetime_activity
0,0


## 13. Final profile: join contact metadata last

Update: Final profile now includes `final_lifecycle_status`, combining `user_type`, max stage reached, and dormancy timing.


In [17]:
con.execute("""
CREATE OR REPLACE TABLE dev_profile_final_v4 AS
SELECT
    u.developer_id,

    -- Persona
    p.persona,
    p.persona_confidence,
    p.persona_confidence_tier,
    p.persona_entropy,
    p.mixed_persona_flag,
    p.cuda_share,
    p.genai_share,
    p.robotics_share,
    p.simulation_share,
    p.learning_community_share,

    -- Developer effort segmentation
    e.developer_effort_score,
    e.developer_effort_level,
    e.developer_effort_rank,
    e.recency_weight AS effort_recency_weight,

    -- Journey and dormancy
    js.behavior_journey_stage_30d,
    js.behavior_journey_rank_30d,
    js.current_journey_state_30d,
    js.current_journey_rank_30d,

    COALESCE(d.is_activated, 0) AS is_activated,
    d.lifetime_meaningful_weeks,
    d.last_meaningful_week_start,

    DATE_DIFF('day', d.last_meaningful_week_start, CURRENT_DATE)
        AS days_since_last_meaningful_week,

    d.days_since_last_activity,

    COALESCE(d.dormancy_status, 'Unactivated') AS dormancy_status,

    CASE WHEN d.dormancy_status = 'Dormant' THEN 1 ELSE 0 END AS dormant_flag,
    CASE WHEN d.dormancy_status = 'At_Risk' THEN 1 ELSE 0 END AS at_risk_flag,
    CASE WHEN d.dormancy_status = 'Cooling' THEN 1 ELSE 0 END AS cooling_flag,

    CASE
        WHEN COALESCE(lf.lifetime_activity_count, 0) = 0 THEN 'Unactivated'
        WHEN lf.user_type = 'tourist' THEN 'Tourist'
        WHEN lf.user_type = 'free_email_user' THEN 'FreeEmail'
        WHEN COALESCE(
            d.days_since_last_activity,
            DATE_DIFF('day', d.last_meaningful_week_start, CURRENT_DATE)
        ) >= 365 THEN 'Dormant_' || lf.max_stage_reached
        WHEN COALESCE(
            d.days_since_last_activity,
            DATE_DIFF('day', d.last_meaningful_week_start, CURRENT_DATE)
        ) >= 180 THEN 'AtRisk_' || lf.max_stage_reached
        ELSE 'Active_' || lf.max_stage_reached
    END AS final_lifecycle_status,

    -- Recency / incremental windows
    r.* EXCLUDE (developer_id),

    -- Lifetime features
    lf.* EXCLUDE (
        developer_id,
        cuda_score,
        genai_score,
        robotics_score,
        simulation_score,
        learning_community_score,
        other_persona_score
    ),

    -- Contact enrichment
    c.created_date AS contact_created_date,
    c.first_activity_date AS contact_first_activity_date,
    c.last_activity_date AS contact_last_activity_date,
    c.account_id,
    c.account_type,
    c.country,
    c.region,
    c.industry_segment_vertical,
    c.program_application_source,
    c.organization_english_name,
    c.normalized_account_name,
    c.wwfo_category,
    c.wwfo_target_list,

    CASE WHEN c.developer_id IS NULL THEN 1 ELSE 0 END AS missing_contact_metadata_flag

FROM developer_universe_v2 u
LEFT JOIN dev_persona_v2 p USING (developer_id)
LEFT JOIN dev_effort_level_v2 e USING (developer_id)
LEFT JOIN dev_journey_state_v2 js USING (developer_id)
LEFT JOIN dev_dormancy_status_v2 d USING (developer_id)
LEFT JOIN dev_recency_features_v2 r USING (developer_id)
LEFT JOIN dev_features_lifetime_v2 lf USING (developer_id)
LEFT JOIN contact_one_row_v2 c USING (developer_id)
""")

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS unique_developers,
    AVG(developer_effort_score) AS avg_developer_effort_score,
    SUM(missing_contact_metadata_flag) AS missing_contact_metadata
FROM dev_profile_final_v4
""").fetchdf())

display(con.execute("""
SELECT
    persona,
    current_journey_state_30d,
    dormancy_status,
    final_lifecycle_status,
    COUNT(*) AS developers
FROM dev_profile_final_v4
GROUP BY 1, 2, 3, 4
ORDER BY developers DESC
LIMIT 40
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,unique_developers,avg_developer_effort_score,missing_contact_metadata
0,9379190,9379190,0.427173,475993.0


,persona,current_journey_state_30d,dormancy_status,final_lifecycle_status,developers
0,CUDA,Dormant_Historically_Active,Dormant,Tourist,1512053
1,Unknown,Unactivated,Unactivated,Unactivated,1293081
2,GenAI,Dormant_Historically_Active,Dormant,Tourist,834750
3,CUDA,Dormant_Historically_Active,Dormant,Dormant_Build,706717
4,GenAI,At_Risk_Historically_Active,At_Risk,Tourist,412258
5,Unknown,Dormant_Historically_Active,Dormant,Tourist,392175
6,CUDA,At_Risk_Historically_Active,At_Risk,Tourist,352672
7,Simulation,Dormant_Historically_Active,Dormant,Tourist,323800
8,GenAI,Unactivated,Unactivated,Unactivated,282343
9,Learning_Community,Dormant_Historically_Active,Dormant,Tourist,229123


## 15. Basic EDA of final developer profile

These cells give the lightweight final-data EDA needed before modeling: final row coverage, effort-level distribution, lifecycle distribution, segment quality by geography/account type, activity mix by effort level, and extreme outlier checks.


In [18]:

print('Final profile shape and coverage')
display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    SUM(CASE WHEN missing_contact_metadata_flag = 1 THEN 1 ELSE 0 END) AS missing_contact_metadata,
    ROUND(100.0 * SUM(CASE WHEN missing_contact_metadata_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_contact_metadata_pct,
    MIN(lifetime_first_activity_date) AS min_lifetime_first_activity_date,
    MAX(lifetime_last_activity_date) AS max_lifetime_last_activity_date
FROM dev_profile_final_v4
""").fetchdf())

print('Effort level summary')
display(con.execute("""
SELECT
    developer_effort_level,
    developer_effort_rank,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_developers,
    ROUND(AVG(developer_effort_score), 3) AS avg_effort_score,
    ROUND(AVG(lifetime_activity_count), 2) AS avg_lifetime_activity_count,
    ROUND(AVG(lifetime_activity_score_sum), 2) AS avg_lifetime_activity_score_sum,
    ROUND(AVG(lifetime_unique_activity_types), 2) AS avg_lifetime_activity_types,
    ROUND(AVG(lifetime_unique_modalities), 2) AS avg_lifetime_modalities
FROM dev_profile_final_v4
GROUP BY 1,2
ORDER BY developer_effort_rank DESC
""").fetchdf())

print('Activity mix by effort level')
display(con.execute("""
SELECT
    developer_effort_level,
    COUNT(*) AS developers,
    ROUND(AVG(lifetime_dli_training_count), 2) AS avg_dli_training,
    ROUND(AVG(lifetime_webinar_count), 2) AS avg_webinars,
    ROUND(AVG(lifetime_devzone_download_count), 2) AS avg_devzone_downloads,
    ROUND(AVG(lifetime_ngc_download_count), 2) AS avg_ngc_downloads,
    ROUND(AVG(lifetime_api_count), 2) AS avg_api,
    ROUND(AVG(lifetime_forum_count), 2) AS avg_forum,
    ROUND(AVG(lifetime_bug_count), 2) AS avg_bugs,
    ROUND(AVG(lifetime_hackathon_count), 2) AS avg_hackathons
FROM dev_profile_final_v4
GROUP BY 1, developer_effort_rank
ORDER BY developer_effort_rank DESC
""").fetchdf())

print('Lifecycle and effort cross-tab')
display(con.execute("""
SELECT
    developer_effort_level,
    final_lifecycle_status,
    COUNT(*) AS developers
FROM dev_profile_final_v4
GROUP BY 1,2, developer_effort_rank
ORDER BY developer_effort_rank DESC, developers DESC
LIMIT 50
""").fetchdf())

print('Top outliers by raw activity count')
display(con.execute("""
SELECT
    developer_id,
    developer_effort_level,
    developer_effort_score,
    lifetime_activity_count,
    lifetime_activity_score_sum,
    lifetime_unique_activity_types,
    lifetime_devzone_download_count,
    lifetime_ngc_download_count,
    lifetime_api_count,
    final_lifecycle_status,
    country,
    account_type,
    normalized_account_name
FROM dev_profile_final_v4
ORDER BY lifetime_activity_count DESC
LIMIT 25
""").fetchdf())


Final profile shape and coverage


,rows,developers,missing_contact_metadata,missing_contact_metadata_pct,min_lifetime_first_activity_date,max_lifetime_last_activity_date
0,9379190,9379190,475993.0,5.07,2020-01-01,2026-03-12


Effort level summary


,developer_effort_level,developer_effort_rank,developers,pct_developers,avg_effort_score,avg_lifetime_activity_count,avg_lifetime_activity_score_sum,avg_lifetime_activity_types,avg_lifetime_modalities
0,very high effort,4,766034,8.17,1.462,42.89,137.27,2.21,1.74
1,high effort,3,1151292,12.27,0.716,19.25,73.98,2.50,1.97
2,medium effort,2,2152014,22.94,0.468,3.62,16.28,2.19,1.93
3,low effort,1,3590938,38.29,0.294,1.82,3.65,1.58,1.45
4,no activity,0,1718912,18.33,0.000,0.00,0.00,0.00,0.00


Activity mix by effort level


,developer_effort_level,developers,avg_dli_training,avg_webinars,avg_devzone_downloads,avg_ngc_downloads,avg_api,avg_forum,avg_bugs,avg_hackathons
0,very high effort,766034,0.55,0.11,23.97,9.19,5.70,0.75,0.10,0.0
1,high effort,1151292,0.63,0.09,13.65,0.52,1.27,0.13,0.03,0.0
2,medium effort,2152014,0.36,0.02,1.77,0.02,0.03,0.01,0.00,0.0
3,low effort,3590938,0.00,0.02,0.52,0.01,0.16,0.01,0.00,0.0
4,no activity,1718912,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0


Lifecycle and effort cross-tab


,developer_effort_level,final_lifecycle_status,developers
0,very high effort,Tourist,346425
1,very high effort,Active_Build,166225
2,very high effort,Active_Discover,103889
3,very high effort,Active_Learn,70478
4,very high effort,Active_Evaluate,20593
5,very high effort,Dormant_Build,19369
6,very high effort,Active_Champion,15655
7,very high effort,Dormant_Champion,8274
8,very high effort,AtRisk_Build,7216
9,very high effort,FreeEmail,4065


Top outliers by raw activity count


,developer_id,developer_effort_level,developer_effort_score,lifetime_activity_count,lifetime_activity_score_sum,lifetime_unique_activity_types,lifetime_devzone_download_count,lifetime_ngc_download_count,lifetime_api_count,final_lifecycle_status,country,account_type,normalized_account_name
0,5903679,very high effort,5.344300,2027375,10015777.0,2,0.0,2027374.0,0.0,Active_Champion,UNITED STATES,enterprise,NVIDIA
1,5886356,very high effort,5.113222,789400,3946993.0,4,1.0,789395.0,0.0,Active_Build,SINGAPORE,enterprise,Straitdeer Pte Ltd
2,3931685,very high effort,4.915595,187095,935275.0,1,0.0,187095.0,0.0,Active_Build,PORTUGAL,startup,Onfido
3,1303597,very high effort,4.779564,151947,450792.0,3,151937.0,0.0,0.0,Active_Build,NaN,NaN,NaN
4,2028858,very high effort,4.740226,142980,714580.0,1,0.0,142980.0,0.0,Active_Build,NaN,NaN,NaN
5,6309430,very high effort,4.593052,121950,235235.0,5,656.0,121180.0,102.0,Active_Champion,UNITED STATES,enterprise,NVIDIA
6,4634152,very high effort,1.937231,107089,180756.0,1,0.0,107089.0,0.0,Active_Champion,UNITED STATES,unknown,Unclassified - Invalid
7,6125204,very high effort,4.257735,68992,344961.0,9,31.0,68888.0,0.0,Active_Build,THAILAND,unknown,Not Normalized
8,4833930,very high effort,1.001874,55061,275414.0,2,0.0,55060.0,0.0,Dormant_Build,UNITED STATES,enterprise,NVIDIA
9,225774,high effort,0.981601,53677,268041.0,1,0.0,53677.0,0.0,Dormant_Build,UNITED STATES,enterprise,Paige ai


## Effort mapping QA

These checks show how often the AI mapping is used, which activities are most misaligned with the original activity score, and how much low/medium-confidence effort evidence exists. The misalignment features are retained because they help distinguish business value from actual user effort.


In [19]:
print('Score-effort misalignment summary')
display(con.execute("""
SELECT
    activity,
    effort_level,
    ai_effort_match_source,
    ai_confidence,
    COUNT(*) AS rows,
    ROUND(AVG(activity_score), 2) AS avg_activity_score,
    ROUND(AVG(effort_rank), 2) AS avg_effort_rank,
    ROUND(AVG(score_effort_gap), 2) AS avg_score_effort_gap,
    SUM(score_effort_misalignment_flag) AS misaligned_rows
FROM activity_labeled_v2
GROUP BY 1,2,3,4
ORDER BY misaligned_rows DESC, rows DESC
LIMIT 30
""").fetchdf())

print('Developer-level effort feature QA')
display(con.execute("""
SELECT
    developer_effort_level,
    COUNT(*) AS developers,
    ROUND(AVG(lifetime_avg_effort_rank), 3) AS avg_row_effort_rank,
    ROUND(AVG(lifetime_total_confidence_weighted_effort), 2) AS avg_total_confidence_effort,
    ROUND(AVG(lifetime_score_effort_misalignment_count), 2) AS avg_misalignment_count,
    ROUND(AVG(lifetime_low_medium_confidence_effort_count), 2) AS avg_low_medium_confidence_count
FROM dev_profile_final_v4
GROUP BY 1
ORDER BY AVG(developer_effort_rank) DESC
""").fetchdf())


Score-effort misalignment summary


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,activity,effort_level,ai_effort_match_source,ai_confidence,rows,avg_activity_score,avg_effort_rank,avg_score_effort_gap,misaligned_rows
0,dev program membership,high,event_or_dli_or_file_type,medium,6654555,1.02,3.0,-73.98,6654555.0
1,ngc downloads,high,event_or_dli_or_file_type,high,6016375,5.04,3.0,-69.96,6016375.0
2,ngc downloads,very high,event_or_dli_or_file_type,high,490350,3.98,4.0,-96.02,490350.0
3,conference,high,event_or_dli_or_file_type,high,401433,9.98,3.0,-65.02,401433.0
4,devzone downloads,high,event_or_dli_or_file_type,high,154996,2.03,3.0,-72.97,154996.0
5,brev,high,NaN,NaN,121821,3.00,3.0,-72.00,121821.0
6,bugs filed,high,NaN,NaN,115133,3.00,3.0,-72.00,115133.0
7,dev program membership,moderate,event_or_dli_or_file_type,medium,78646,0.01,2.0,-49.99,78267.0
8,devzone downloads,high,NaN,NaN,75318,3.00,3.0,-72.00,75318.0
9,devzone downloads,moderate,event_or_dli_or_file_type,high,37429076,2.98,2.0,-47.02,12187.0


Developer-level effort feature QA


,developer_effort_level,developers,avg_row_effort_rank,avg_total_confidence_effort,avg_misalignment_count,avg_low_medium_confidence_count
0,very high effort,766034,2.142,79.61,9.24,2.65
1,high effort,1151292,2.206,35.47,1.66,1.14
2,medium effort,2152014,2.283,7.10,1.03,0.97
3,low effort,3590938,2.203,3.09,0.81,0.81
4,no activity,1718912,0.000,0.00,0.00,0.00


## 14. Feature validation checklist

This validates final profile shape, feature ranges, journey/dormancy separation, and dictionary determinism before modeling.


In [20]:
validation = con.execute("""
WITH dict_check AS (
    SELECT COUNT(*) AS inconsistent_activity_dictionary_labels
    FROM (
        SELECT activity
        FROM activity_dictionary_v2
        GROUP BY activity
        HAVING COUNT(DISTINCT fallback_journey_signal) > 1
            OR COUNT(DISTINCT fallback_effort_level) > 1
            OR COUNT(DISTINCT modality) > 1
    ) x
),
profile_check AS (
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT developer_id) AS distinct_developers,
        SUM(CASE WHEN activity_count_0_30d < 0 OR lifetime_activity_count < 0 THEN 1 ELSE 0 END) AS negative_count_violations,
        SUM(CASE WHEN persona_confidence < 0 OR persona_confidence > 1 THEN 1 ELSE 0 END) AS persona_confidence_violations,
        SUM(CASE WHEN persona_entropy < -0.000001 OR persona_entropy > 1.000001 THEN 1 ELSE 0 END) AS persona_entropy_violations,
        SUM(CASE WHEN has_activity_0_30d = 0 AND activity_count_0_30d <> 0 THEN 1 ELSE 0 END) AS zero_activity_flag_violations,
        SUM(CASE WHEN recent_build_flag = 1 AND build_count_0_30d = 0 THEN 1 ELSE 0 END) AS recent_build_flag_violations,
        SUM(CASE WHEN missing_contact_metadata_flag NOT IN (0,1) THEN 1 ELSE 0 END) AS contact_flag_violations,
        SUM(CASE WHEN developer_effort_score < 0 THEN 1 ELSE 0 END) AS developer_effort_score_violations,
        SUM(CASE WHEN lifetime_avg_effort_rank < 0 OR lifetime_avg_effort_rank > 4 THEN 1 ELSE 0 END) AS lifetime_avg_effort_rank_violations,
        SUM(CASE WHEN lifetime_total_confidence_weighted_effort < 0 THEN 1 ELSE 0 END) AS lifetime_confidence_effort_violations,
        SUM(CASE WHEN developer_effort_rank NOT BETWEEN 0 AND 4 THEN 1 ELSE 0 END) AS developer_effort_rank_violations,
        SUM(CASE WHEN behavior_journey_stage_30d = 'Unactivated' AND lifetime_activity_count > 0 THEN 1 ELSE 0 END) AS unactivated_lifetime_activity_violations,
        SUM(CASE WHEN final_lifecycle_status = 'Unactivated' AND lifetime_activity_count > 0 THEN 1 ELSE 0 END) AS final_unactivated_lifetime_activity_violations,
        SUM(CASE WHEN lifetime_activity_count = 0 AND final_lifecycle_status <> 'Unactivated' THEN 1 ELSE 0 END) AS zero_lifetime_not_unactivated_violations
    FROM dev_profile_final_v4
)
SELECT *
FROM profile_check
CROSS JOIN dict_check
""").fetchdf()

display(validation)

if validation.loc[0, "rows"] != validation.loc[0, "distinct_developers"]:
    raise ValueError("Final table is not one row per developer")

violation_cols = [c for c in validation.columns if c.endswith("violations") or c == "inconsistent_activity_dictionary_labels"]
violations = validation.loc[0, violation_cols].sum()
if violations > 0:
    print("WARNING: validation violations found. Inspect the table above before modeling.")
else:
    print("All feature validation checks passed.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,distinct_developers,negative_count_violations,persona_confidence_violations,persona_entropy_violations,zero_activity_flag_violations,recent_build_flag_violations,contact_flag_violations,developer_effort_score_violations,lifetime_avg_effort_rank_violations,lifetime_confidence_effort_violations,developer_effort_rank_violations,unactivated_lifetime_activity_violations,final_unactivated_lifetime_activity_violations,zero_lifetime_not_unactivated_violations,inconsistent_activity_dictionary_labels
0,9379190,9379190,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


All feature validation checks passed.


## 15. Table inventory

In [21]:
final_tables = [
    "activity_base_v2",
    "contact_one_row_v2",
    "activity_labeled_v2",
    "developer_universe_v2",
    "dev_features_0_30d_v2",
    "dev_features_30_90d_v2",
    "dev_features_90_180d_v2",
    "dev_recency_features_v2",
    "dev_features_lifetime_v2",
    "dev_effort_level_v2",
    "dev_contact_persona_v2",
    "dev_persona_v2",
    "dev_journey_state_v2",
    "dev_weekly_features_v2",
    "dev_meaningful_week_v2",
    "dev_activation_v2",
    "dev_dormancy_status_v2",
    "dev_profile_final_v4",
]

inventory = []
for t in final_tables:
    exists = con.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = ?", [t]).fetchone()[0] > 0
    rows = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] if exists else None
    inventory.append({"table": t, "rows": rows})

display(pd.DataFrame(inventory))


,table,rows
0,activity_base_v2,69347501
1,contact_one_row_v2,8903197
2,activity_labeled_v2,69347501
3,developer_universe_v2,9379190
4,dev_features_0_30d_v2,9379190
5,dev_features_30_90d_v2,9379190
6,dev_features_90_180d_v2,9379190
7,dev_recency_features_v2,9379190
8,dev_features_lifetime_v2,9379190
9,dev_effort_level_v2,9379190


In [22]:
# Close when finished.
con.close()